In [ ]:
import pandas as pd
import math
import re
import numpy as np
import matplotlib.pyplot as plt
import gc
import time
import random
import os
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupShuffleSplit, cross_val_score
from sklearn.utils import shuffle
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

from sklearn.metrics import classification_report,confusion_matrix
from sklearn.model_selection import RandomizedSearchCV

from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import mcnemar

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
print(f"({round(882/1638, 2)}:({round(756/1638,2)})")

Generate duplicates of Accepted Statements for Variant generations

In [ ]:
# df = pd.read_csv('accepted_dataset - accepted_statements.csv')
# df_dict = df.to_dict()

In [ ]:
# df_dict.keys()

In [ ]:
# content_list = df_dict.get('content')
# cue_list = df_dict.get('cue')
# labels_list = df_dict.get('labels')
# reviewid_list = df_dict.get('reviewId')

In [ ]:
# new_content_list = []
# new_cue_list = []
# new_labels_list = []
# new_reviewid_list = []

In [ ]:
# for i in range(len(content_list)):
#     for j in range(6):
#         new_content_list.append(content_list[i])
#         new_cue_list.append(cue_list[i])
#         new_labels_list.append(labels_list[i])
#         new_reviewid_list.append(reviewid_list[i])

In [ ]:
# df_dict = {
#     'content': new_content_list,
#     'cue': new_cue_list,
#     'labels': new_labels_list,
#     'reviewId': new_reviewid_list
# }
# df = pd.DataFrame(df_dict)


In [ ]:
# df['original_text'] = df['content']

In [ ]:
#df.to_excel('accepted_dataset-variants_template.xlsx', index=False)

Check Final Variant Dataset for duplicates

In [ ]:
df = pd.read_csv("accepted_dataset - Accepted Variants.csv")

In [ ]:
df.shape

In [ ]:
df.head(2)

In [ ]:
df.info()

In [ ]:
df["text_clean"] = (
    df["content"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True) # remove symbols/punctuation
    .str.replace(r"_", " ", regex=True) # remove underscores
    .str.replace(r"[^\x00-\x7F]+", "", regex=True)  # remove emojis/non-ASCII
    .str.replace(r"\s+", " ", regex=True) # collapse multiple spaces
    .str.strip()
)

In [ ]:
df.sample(2)

In [ ]:
df[df.duplicated(subset=['text_clean'], keep=False)]

#Start Modelling

In [ ]:
RANDOM_SEED = 42

In [ ]:
#re initialize accepted variants
df = pd.read_csv('final_accepted_dataset - Accepted Variants.csv')


In [ ]:
import re
orig = df.loc[df['position'].notna(), ['reviewId','original_text']].drop_duplicates('reviewId')
pats = {
    'email': r'[\w\.-]+@[\w\.-]+\.\w+',
    'phone': r'(?:\+?62|0)8\d{7,11}',
    'url':   r'https?://\S+|www\.\S+',
    'handle': r'(?<!\w)@\w{3,}',
    'long_number': r'\d{8,}',
}
for name, pat in pats.items():
    hit = orig[orig['original_text'].str.contains(pat, regex=True, na=False)]
    print(name, len(hit))
    if len(hit): print(hit['original_text'].head(10).to_string())

In [ ]:
def set_reproducibility_seed(seed=RANDOM_SEED):
    """
    Forces all random number generators across Python, NumPy,
    and PyTorch backends to lock onto a deterministic baseline.
    """
    # 1. Standard Python environment seeds
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # 2. Mathematical matrix libraries
    np.random.seed(seed)

    # 3. PyTorch Core (CPU and all active GPUs)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Safely handles multi-GPU setups if applicable

    # 4. Enforce deterministic algorithmic behaviors in CUDA
    # Warning: This can slightly slow down training, but guarantees matching outputs.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"🔒 Global reproducibility seed fixed at: {seed}")

# Execute before training your models
set_reproducibility_seed(42)

In [ ]:
df.shape

In [ ]:
# Clean and convert labels to numeric (int32 or int64)
df['labels'] = df['labels'].str.replace('POSITIF', '1').str.replace('NEGATIF', '0')
df['labels'] = pd.to_numeric(df['labels'])

In [ ]:
df['labels'].value_counts()

In [ ]:
#Shuffle first to simulate real data
df = shuffle(df, random_state=RANDOM_SEED).reset_index(drop=True)

In [ ]:
X = df[['content','position','language','reviewId','is_original']]
y = df['labels']

In [ ]:
X.head()

In [ ]:
#create group for groupshufflesplit
groups = X.reviewId.reset_index(drop=True)

In [ ]:
#Split 80:20
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [ ]:
X_test.head(12)

In [ ]:
y_test.head(12)

In [ ]:
X_test[X_test.reviewId == 'cb4fb0b4-b3d2-41a8-87e0-087057bf597f']
#X_test[X_test.content == 'Terasa seperti punya travel agent sendiri.cool']

In [ ]:
#check split if split has leakage
train_groups = set(groups[train_idx])
test_groups = set(groups[test_idx])

overlap = train_groups & test_groups  # intersection

if overlap:
    print(f"LEAKAGE DETECTED: {len(overlap)} groups appear in both splits")
    print(overlap)
else:
    print("No overlap — splits are clean")

In [ ]:
tr = X_train.copy(); tr['label'] = y_train.values; tr['split'] = 'train'
te = X_test.copy();  te['label'] = y_test.values;  te['split'] = 'test'
al = pd.concat([tr, te]); al['label'] = al['label'].map({0:'Negative', 1:'Positive'})
print(al.groupby('split')['reviewId'].nunique())
print(pd.crosstab(al['split'], al['label']))
print(pd.crosstab([al['split'], al['label']], al['position']))
print(pd.crosstab([al['split'], al['label']], al['language']))
print(al.groupby(['split','label'])['reviewId'].nunique())

In [ ]:
#Preprocess before pipeline
def preprocess_text(text):
    #lowercase and strip whitespace in front/back
    text = text.lower().strip()

    #removal of non ASCII characters
    text = re.sub(r"[^\x00-\x7F]+", "", text)

    #remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
#only get the content column to fit to the model
X_train_statements = X_train['content']
X_test_statements = X_test['content']

In [ ]:
def check_error_from_predictions(predictions):
  # Map predictions back to the structured evaluation dataframe
  results_df = X_test.copy().reset_index(drop=True)
  results_df['labels'] = y_test.reset_index(drop=True)
  results_df['predicted_label'] = predictions
  # Identify exactly where predictions diverged from true labels
  results_df['is_error'] = results_df['labels'] != results_df['predicted_label']

  f1_indo = f1_score(results_df[results_df['language']=='Indonesian']['labels'], results_df[results_df['language']=='Indonesian']['predicted_label'], average='macro')
  f1_eng = f1_score(results_df[results_df['language']=='English']['labels'], results_df[results_df['language']=='English']['predicted_label'], average='macro')
  #lang_sensitivity = abs(f1_indo - f1_eng)
  print(f"Macro F1 Indo: {f1_indo:.2f}")
  print(f"Macro F1 Eng: {f1_eng:.2f}")

  # 2. Compute Position Sensitivity Score
  f1_beg = f1_score(results_df[results_df['position']=='Beginning']['labels'], results_df[results_df['position']=='Beginning']['predicted_label'], average='macro')
  f1_mid = f1_score(results_df[results_df['position']=='Middle']['labels'], results_df[results_df['position']=='Middle']['predicted_label'], average='macro')
  f1_end = f1_score(results_df[results_df['position']=='End']['labels'], results_df[results_df['position']=='End']['predicted_label'], average='macro')
  #pos_sensitivity = max(f1_beg, f1_mid, f1_end) - min(f1_beg, f1_mid, f1_end)

  print(f"Macro F1 Beginning: {f1_beg:.2f}")
  print(f"Macro F1 Middle: {f1_mid:.2f}")
  print(f"Macro F1 End: {f1_end:.2f}" )

  #Error Breakdown
  # Group errors by sentence-internal position (Beginning, Middle, End)
  position_summary = results_df.groupby('position').agg(
        Total_Samples=('is_error', 'count'),
        Total_Errors=('is_error', 'sum'),
        Error_Rate=('is_error', 'mean')
  ).reset_index()
  position_summary['Error_Rate'] = position_summary['Error_Rate'].map('{:.3f}'.format)
  print("\n[Position Sensitivity Summary]")
  print(position_summary.to_string(index=False))

  # Group errors by sentiment cue language (Indonesian/English)
  language_summary = results_df.groupby('language').agg(
        Total_Samples=('is_error', 'count'),
        Total_Errors=('is_error', 'sum'),
        Error_Rate=('is_error', 'mean')
  ).reset_index()
  language_summary['Error_Rate'] = language_summary['Error_Rate'].map('{:.3f}'.format)
  print("\n[Language Sensitivity Summary]")
  print(language_summary.to_string(index=False))

  total_groups = results_df['reviewId'].nunique()
  consistent_groups = 0
  total_flips = 0
  total_comparisons = 0

  for review_id, group in results_df.groupby('reviewId'):
      #print(group)
      original_text = group[group['is_original'] == 'Y']
      scalar_label = original_text.index[0]
      positional_index = group.index.get_loc(scalar_label)
      preds = group['predicted_label'].values

      # 1. Check Consistency: Are all 6 predictions identical?
      if len(set(preds)) == 1:
          consistent_groups += 1

      # 2. Compute Flip Rate: Count transitions between variants
      # (e.g., comparing each variant to the original baseline variant)
      base_pred = preds[positional_index]
      for i, p in enumerate(preds):

        # Dynamically bypass the base prediction index (whether it is 0, 5, or anything else)
        if i == positional_index:
            continue

        total_comparisons += 1
        if p != base_pred:
            total_flips += 1

  consistency_rate = consistent_groups / total_groups
  flip_rate = total_flips / total_comparisons

  print(f"\nPrediction Consistency Rate: {consistency_rate:.3f}")
  print(f"\nPrediction Flip Rate: {flip_rate:.3f}")

  # Combines structural position and language to isolate joint structural feature impacts
  interaction_matrix = results_df.groupby(['position', 'language']).agg(
      Samples=('is_error', 'count'),
      Errors=('is_error', 'sum'),
      Error_Rate=('is_error', 'mean')
  ).reset_index()

  # Calculate a matching accuracy column for clear visibility
  interaction_matrix['Accuracy_Percentage'] = 100 - interaction_matrix['Error_Rate']

  # Format metrics cleanly for direct markdown/LaTeX table exports
  interaction_matrix['Error_Rate'] = interaction_matrix['Error_Rate'].map('{:.3f}'.format)
  interaction_matrix['Accuracy_Percentage'] = interaction_matrix['Accuracy_Percentage'].map('{:.2f}%'.format)

  print(f"\n${interaction_matrix.to_string(index=False)}")

  #print macro f1score for each position+language
  f1_beg_ind = f1_score(results_df[(results_df['position'] == 'Beginning') & (results_df['language'] == 'Indonesian')]['labels'], results_df[(results_df['position'] == 'Beginning') & (results_df['language'] == 'Indonesian')]['predicted_label'], average='macro')
  f1_mid_ind = f1_score(results_df[(results_df['position'] == 'Middle') & (results_df['language'] == 'Indonesian')]['labels'], results_df[(results_df['position'] == 'Middle') & (results_df['language'] == 'Indonesian')]['predicted_label'], average='macro')
  f1_end_ind = f1_score(results_df[(results_df['position'] == 'End') & (results_df['language'] == 'Indonesian')]['labels'], results_df[(results_df['position'] == 'End') & (results_df['language'] == 'Indonesian')]['predicted_label'], average='macro')
  f1_beg_eng = f1_score(results_df[(results_df['position'] == 'Beginning') & (results_df['language'] == 'English')]['labels'], results_df[(results_df['position'] == 'Beginning') & (results_df['language'] == 'English')]['predicted_label'], average='macro')
  f1_mid_eng = f1_score(results_df[(results_df['position'] == 'Middle') & (results_df['language'] == 'English')]['labels'], results_df[(results_df['position'] == 'Middle') & (results_df['language'] == 'English')]['predicted_label'], average='macro')
  f1_end_eng = f1_score(results_df[(results_df['position'] == 'End') & (results_df['language'] == 'English')]['labels'], results_df[(results_df['position'] == 'End') & (results_df['language'] == 'English')]['predicted_label'], average='macro')

  print(f"Macro F1 Beginning Ind: {f1_beg_ind:.2f}")
  print(f"Macro F1 Middle Ind: {f1_mid_ind:.2f}")
  print(f"Macro F1 End Ind: {f1_end_ind:.2f}" )
  print(f"Macro F1 Beginning En: {f1_beg_eng:.2f}")
  print(f"Macro F1 Middle En: {f1_mid_eng:.2f}")
  print(f"Macro F1 End En: {f1_end_eng:.2f}" )


  # --- 4. EXPORT TARGETED ERROR ROWS FOR MANUAL INSPECTION ---
  # Captures sample text sequences to diagnose failure modes (e.g., tokenization flaws)
  error_samples = results_df[results_df['is_error'] == True][['content', 'position', 'language', 'reviewId', 'labels', 'predicted_label']]

  return interaction_matrix, error_samples

In [ ]:
def build_mcnemar_table(df, model_name, condition_col, cond_a, cond_b):
  model_df = df

  # Dynamically set index columns based on the condition being compared
  if condition_col == 'position':
      index_cols = ['reviewId', 'language']
  elif condition_col == 'language':
      index_cols = ['reviewId', 'position']
  else:
      raise ValueError(f"Invalid condition_col: {condition_col}")

  a = model_df[model_df[condition_col] == cond_a].set_index(index_cols)['correct']
  b = model_df[model_df[condition_col] == cond_b].set_index(index_cols)['correct']
  paired = pd.concat([a,b], axis=1, keys=[cond_a, cond_b]).dropna()

  n_both_correct = ((paired[cond_a] == 1) & (paired[cond_b] == 1)).sum()
  n_both_incorrect = ((paired[cond_a] == 0) & (paired[cond_b] == 0)).sum()
  n_a_only_correct = ((paired[cond_a] == 1) & (paired[cond_b] == 0)).sum()
  n_b_only_correct = ((paired[cond_a] == 0) & (paired[cond_b] == 1)).sum()

  return [[n_both_correct, n_a_only_correct], [n_b_only_correct, n_both_incorrect]]

def do_mcnemar_test(predictions, model_name):
  df_for_check = X_test.copy().reset_index(drop=True)
  df_for_check['labels'] = y_test.reset_index(drop=True)
  df_for_check['predicted_label'] = predictions
  df_for_check['correct'] = (df_for_check['labels'] == df_for_check['predicted_label']).astype(int)

  position_pairs = [('Beginning','Middle'),('Middle','End'),('Beginning','End')]
  language_pairs = [('Indonesian','English')]

  results = []
  for cond_a, cond_b in position_pairs:
    table = build_mcnemar_table(df_for_check, model_name, 'position', cond_a, cond_b)
    # Use exact binomial version when discordant pair count is small (rule of thumb: < 25)
    n_disc = table[0][1] + table[1][0]
    result = mcnemar(table, exact=(n_disc < 25), correction=True)
    results.append({'model':model_name, 'comparison':f'{cond_a} vs {cond_b}','discordant_pairs':n_disc, 'statistic':result.statistic,'p_value':result.pvalue})

  for cond_a, cond_b in language_pairs:
    table = build_mcnemar_table(df_for_check, model_name, 'language', cond_a, cond_b)
    # Use exact binomial version when discordant pair count is small (rule of thumb: < 25)
    n_disc = table[0][1] + table[1][0]
    result = mcnemar(table, exact=(n_disc < 25), correction=True)
    results.append({'model':model_name, 'comparison':f'{cond_a} vs {cond_b}','discordant_pairs':n_disc, 'statistic':result.statistic,'p_value':result.pvalue})
  results_df = pd.DataFrame(results)
  results_df['p_adjusted'] = multipletests(results_df['p_value'], method='holm')[1]
  results_df['significant'] = results_df['p_adjusted'] < 0.05
  return results_df

In [ ]:
#create TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    preprocessor=preprocess_text,
    stop_words=None,
    ngram_range=(1,2),
    max_df=0.95,
)

#Logistic Regression

In [ ]:
#make a pipeline for Logistic Regression
lr_pipeline = Pipeline([
    ('tfidf', tfidf_vectorizer),
    ('model', LogisticRegression(max_iter=1000, solver='lbfgs', C=1))
])

In [ ]:
lr_pipeline.fit(X_train_statements, y_train)
lr_preds = lr_pipeline.predict(X_test_statements)

In [ ]:
print(classification_report(y_test, lr_preds))

In [ ]:
lr_pipeline.score(X_test['content'], y_test)

In [ ]:
ax = sns.heatmap(confusion_matrix(y_test, lr_preds), annot=True, fmt='.0f',
                 xticklabels=['Negative', 'Positive'], yticklabels=['Negative','Positive'],
                 cmap='Blues')
ax.set(xlabel="Predicted Label", ylabel="True Label")
ax.set(title="Logistic Regression");

In [ ]:
interaction_matrix, error_samples = check_error_from_predictions(lr_preds)

In [ ]:
lr_mcnemar_result = do_mcnemar_test(lr_preds, 'Logistic Regression')

In [ ]:
interaction_matrix

In [ ]:
error_samples

In [ ]:
lr_mcnemar_result

#LinearSVC

In [ ]:
#Make SVM pipeline
svm_pipeline = Pipeline([
    ('tfidf', tfidf_vectorizer),
    ('model', LinearSVC(C=1, max_iter=1000, class_weight='balanced'))
])

In [ ]:
svm_pipeline.fit(X_train_statements, y_train)
svm_preds = svm_pipeline.predict(X_test_statements)

In [ ]:
print(classification_report(y_test, svm_preds))

In [ ]:
#accuracy score
svm_pipeline.score(X_test['content'], y_test)

In [ ]:
ax = sns.heatmap(confusion_matrix(y_test, svm_preds), annot=True, fmt='.0f',
                 xticklabels=['Negative', 'Positive'], yticklabels=['Negative','Positive'],
                 cmap='Blues')
ax.set(xlabel="Predicted Label", ylabel="True Label")
ax.set(title="LinearSVC");

In [ ]:
interaction_matrix, error_samples = check_error_from_predictions(svm_preds)

In [ ]:
svm_mcnemar_result = do_mcnemar_test(svm_preds, 'Linear SVC')

In [ ]:
interaction_matrix

In [ ]:
error_samples

In [ ]:
svm_mcnemar_result

#TRANSFORMER MODELS

In [ ]:
# Reduce batch size to save memory
BATCH_SIZE = 16
MAX_SEQ_LEN = 60
RANDOM_SEED = 42
EPOCHS = 3
LEARNING_RATE = 1e-5

In [ ]:
# #check maximum length of text after tokenization
# check_max_len = 0
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
# # For every sentence...
# for sent in X_train_statements:

#     # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
#     input_ids = tokenizer.encode(sent, add_special_tokens=True)

#     # Update the maximum sentence length.
#     check_max_len = max(check_max_len, len(input_ids))

# print('Max sentence length: ', check_max_len)

In [ ]:
#method to change data frame to data loader to load to pytorch model
def df_to_dataloader(text_df, label_df, tokenizer, batch_size=BATCH_SIZE, is_train=False):
  text_list = text_df.tolist()
  label_list = label_df.tolist()

  dataset = list_to_tokenized_dataset(text_list, label_list, tokenizer)

  if is_train:
      # Replaces random shuffle loops with standard training samplers
      sampler = RandomSampler(dataset)
  else:
      sampler = SequentialSampler(dataset)

  data_loader = DataLoader(
      dataset,
      batch_size=batch_size,
      sampler=sampler,
  )
  return data_loader

def list_to_tokenized_dataset(text_list, label_list, tokenizer, max_length: int = MAX_SEQ_LEN):
    """
    A unified method to tokenize text for either BERT or RoBERTa.
    """

    # 2. Tokenize.
    # The tokenizer automatically handles WordPiece vs BPE, casing, and special tokens.
    encoding = tokenizer(
        text_list,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',       # Fixed-length vectors guarantee alignment
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'         # Native PyTorch Tensors
    )

    # Return a TensorDataset. The tokenizer's output 'input_ids' and 'attention_mask'
    # are already (num_samples, max_length). The DataLoader will handle batching.
    return torch.utils.data.TensorDataset(
        encoding['input_ids'],
        encoding['attention_mask'],
        torch.tensor(label_list, dtype=torch.long)
    )

In [ ]:
#reusable method for training model
def train_model(model, train_loader, learning_rate=LEARNING_RATE, epochs=EPOCHS):
    # Route model graph to the GPU
    model = model.to(device)

    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    print("Beginning training phase...")
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        total_train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            # Unpack batch as a tuple (input_ids, attention_mask, labels)
            input_ids, attention_mask, labels = [item.to(device) for item in batch]

            # Forward Pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            total_train_loss += loss.item()

            # Backward Pass & Gradient Clipping to preserve baseline structural weights
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        avg_train_loss = total_train_loss / len(train_loader)
        elapsed = time.time() - start_time
        print(f"Epoch {epoch + 1}/{epochs} | Avg Training Loss: {avg_train_loss:.4f} | Time: {elapsed:.1f}s")

    return model

In [ ]:
#reusable method for evaluating model
def evaluate_model(model, test_loader):
    model = model.to(device)

    model.eval()
    all_predictions = []
    all_true_labels = []

    print("\nEvaluating on Test Set...")
    with torch.no_grad():
        for batch in test_loader:
            # Unpack batch as a tuple (input_ids, attention_mask, labels)
            input_ids, attention_mask, labels = [item.to(device) for item in batch]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            predictions = torch.argmax(logits, dim=1)

            # Move data back to CPU for metrics tracking
            all_predictions.extend(predictions.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    # Calculate performance metrics
    accuracy = accuracy_score(all_true_labels, all_predictions)
    print(f"\nOverall Test Accuracy: {accuracy:.4f}")
    print("\nClassification Report (Condition & Feature Breakdown):")
    print(classification_report(all_true_labels, all_predictions))

    return all_true_labels, all_predictions

In [ ]:
def find_target_token_indices(tokens, target_word):
    """
    BERT's wordpiece tokenizer splits words into subword pieces of
    arbitrary granularity -- e.g. "aplikasi" might become
    ['ap', '##lika', '##si'] and "aplikasinya" might become
    ['ap', '##lika', '##sin', '##ya']. A single piece rarely contains
    the whole target word, so this groups consecutive wordpieces into
    whole words first (a new word starts at any token that does NOT
    begin with '##'), reconstructs each word's surface form, and
    returns the token indices for any word containing target_word as
    a substring. This way "aplikasi" and "aplikasinya" both match when
    target_word="aplikasi", regardless of how finely either was split.

    Print `tokens` for one example first if you want to sanity-check
    the actual split before trusting this.
    """
    target_word = target_word.lower()
    indices = []
    i = 0
    n = len(tokens)
    while i < n:
        group = [i]
        j = i + 1
        while j < n and tokens[j].startswith("##"):
            group.append(j)
            j += 1
        word = "".join(tokens[k].replace("##", "") for k in group)
        if target_word in word.lower():
            indices.extend(group)
        i = j
    return indices


def analyze_attention(text, target_word, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

    target_idx = find_target_token_indices(tokens, target_word)
    if not target_idx:
        print(f"  [!] '{target_word}' not found as a token in: {text}")
        print(f"      Tokens were: {tokens}")
        return None

    with torch.no_grad():
        outputs = model(**inputs)

    # outputs.attentions: tuple of length num_layers,
    # each tensor shaped [batch=1, num_heads, seq_len, seq_len]
    attentions = outputs.attentions
    if attentions is None:
        raise RuntimeError(
            "outputs.attentions is None -- the model did not return attention "
            "weights. Confirm attn_implementation='eager' was set when loading "
            "the model (sdpa/flash attention backends do not expose attention "
            "matrices even with output_attentions=True)."
        )
    num_layers = len(attentions)
    seq_len = attentions[0].shape[-1]

    special_ids = {tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id}
    other_idx = [
        i for i in range(seq_len)
        if i not in target_idx and inputs["input_ids"][0][i].item() not in special_ids
    ]

    cls_idx = 0  # [CLS] is always the first token for BERT

    per_layer_target = []
    per_layer_baseline = []
    per_layer_cls_to_target = []
    per_layer_cls_ratio = []
    per_layer_best_head_ratio = []
    per_layer_best_head_index = []

    for layer_attn_full in attentions:
        layer_attn_full = layer_attn_full[0]  # [num_heads, seq, seq], NOT yet averaged
        num_heads_this_layer = layer_attn_full.shape[0]

        # ---- head-averaged metrics (same as before) ----
        layer_attn_avg = layer_attn_full.mean(dim=0)  # [seq, seq]

        received_avg = layer_attn_avg[:, target_idx].sum(dim=1).mean().item()
        per_layer_target.append(received_avg)

        baseline_avg = layer_attn_avg[:, other_idx].mean().item() if other_idx else float("nan")
        per_layer_baseline.append(baseline_avg)

        cls_to_target_avg = layer_attn_avg[cls_idx, target_idx].sum().item()
        per_layer_cls_to_target.append(cls_to_target_avg)

        # CLS-specific baseline: how much CLS attends to an average other token
        cls_baseline = layer_attn_avg[cls_idx, other_idx].mean().item() if other_idx else float("nan")
        cls_ratio = cls_to_target_avg / max(cls_baseline, 1e-8) if other_idx else float("nan")
        per_layer_cls_ratio.append(cls_ratio)

        # ---- per-head metrics: is any single head anchored, even if the average isn't? ----
        best_ratio = float("-inf")
        best_head = None
        for h in range(num_heads_this_layer):
            head_attn = layer_attn_full[h]  # [seq, seq]
            head_received = head_attn[:, target_idx].sum(dim=1).mean().item()
            head_baseline = head_attn[:, other_idx].mean().item() if other_idx else 0.0
            if other_idx and head_baseline > 1e-8:
                head_ratio = head_received / head_baseline
                if head_ratio > best_ratio:
                    best_ratio = head_ratio
                    best_head = h
        per_layer_best_head_ratio.append(best_ratio if best_head is not None else float("nan"))
        per_layer_best_head_index.append(best_head)

    return {
        "tokens": tokens,
        "target_indices": target_idx,
        "per_layer_target_attention": per_layer_target,
        "per_layer_baseline_attention": per_layer_baseline,
        "per_layer_cls_to_target": per_layer_cls_to_target,
        "per_layer_cls_ratio": per_layer_cls_ratio,
        "per_layer_best_head_ratio": per_layer_best_head_ratio,
        "per_layer_best_head_index": per_layer_best_head_index,
        "overall_target_attention": sum(per_layer_target) / num_layers,
        "overall_baseline_attention": sum(per_layer_baseline) / num_layers,
        "overall_cls_to_target": sum(per_layer_cls_to_target) / num_layers,
        "overall_best_head_ratio": max(per_layer_best_head_ratio),
    }

In [ ]:
def run_check__bert_attention(error_samples, target_word, model, tokenizer):
  model.eval()
  model = model.to(device)
  for index, row in error_samples.iterrows():
    print(f"\n=== {row['position']}-{row['language']}: {row['content']}")
    result = analyze_attention(row['content'], target_word, model, tokenizer)
    if result is None:
      continue

    ratio = result["overall_target_attention"] / max(result["overall_baseline_attention"], 1e-8)
    best_layer = result["per_layer_best_head_ratio"].index(result["overall_best_head_ratio"])
    best_head = result["per_layer_best_head_index"][best_layer]

    print(f"  Avg attention TO 'aplikasi' (all layers, all heads): {result['overall_target_attention']:.4f}")
    print(f"  Avg attention TO other tokens (baseline):            {result['overall_baseline_attention']:.4f}")
    print(f"  Head-averaged ratio (aplikasi / baseline):           {ratio:.2f}x")
    print(f"  Best SINGLE-HEAD ratio found:                        {result['overall_best_head_ratio']:.2f}x "
          f"(layer {best_layer + 1}, head {best_head})")
    print(f"  Avg [CLS] -> 'aplikasi' attention:                   {result['overall_cls_to_target']:.4f}")

    # per-layer breakdown: raw target attention AND the CLS-specific ratio,
    # useful for spotting whether anchoring concentrates in specific layers
    # (Clark et al. found [SEP] anchoring concentrated in layers 6-10)
    print("  Per-layer: attention to 'aplikasi' | CLS ratio | best single-head ratio")
    for i, (v, cls_r, head_r) in enumerate(zip(
        result["per_layer_target_attention"],
        result["per_layer_cls_ratio"],
        result["per_layer_best_head_ratio"],
    ), start=1):
        print(f"    layer {i:2d}: {v:.4f} | CLS ratio {cls_r:6.2f}x | best head {head_r:6.2f}x")

In [ ]:
#Untuk membantu agar tidak crash saat training model nya
# 3. CRITICAL: Clear Pandas from RAM to prevent system crash
# del X_train
# gc.collect()

#BERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
bert_train_dataset = df_to_dataloader(X_train_statements, y_train, tokenizer, is_train=True)
bert_test_dataset = df_to_dataloader(X_test_statements, y_test, tokenizer, is_train=False)

In [ ]:
# CREATE BERT MODEL
classifier = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", # model name
    num_labels = 2, # The number of output labels--2 for binary classification.
    output_attentions = True, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)

In [ ]:
trained_bert_model = train_model(classifier, bert_train_dataset)

In [ ]:
true_labels, bert_pred_labels = evaluate_model(trained_bert_model, bert_test_dataset)

In [ ]:
ax = sns.heatmap(confusion_matrix(true_labels, bert_pred_labels), annot=True, fmt='.0f',
                 xticklabels=['Negative', 'Positive'], yticklabels=['Negative','Positive'],
                 cmap='Blues')
ax.set(xlabel="Predicted Label", ylabel="True Label")
ax.set(title="BERT");

In [ ]:
bert_interaction_matrix, bert_error_samples = check_error_from_predictions(bert_pred_labels)

In [ ]:
bert_mcnemar_result = do_mcnemar_test(bert_pred_labels, 'BERT')

In [ ]:
interaction_matrix

In [ ]:
bert_error_samples

In [ ]:
bert_error_samples.to_excel('bert_error_samples.xlsx', index=False)

In [ ]:
bert_mcnemar_result

In [ ]:
run_check__bert_attention(bert_error_samples, "aplikasi", trained_bert_model, tokenizer)

#mBERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-uncased')

In [ ]:
mbert_train_dataset = df_to_dataloader(X_train_statements, y_train, tokenizer, is_train=True)
mbert_test_dataset = df_to_dataloader(X_test_statements, y_test, tokenizer, is_train=False)

In [ ]:
# CREATE mBERT MODEL
classifier = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-uncased", # model name
    num_labels = 2, # The number of output labels--2 for binary classification.
    output_attentions = True, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)

In [ ]:
trained_mbert_model = train_model(classifier, mbert_train_dataset)

In [ ]:
true_labels, mbert_pred_labels = evaluate_model(trained_mbert_model, mbert_test_dataset)

In [ ]:
ax = sns.heatmap(confusion_matrix(true_labels, mbert_pred_labels), annot=True, fmt='.0f',
                 xticklabels=['Negative', 'Positive'], yticklabels=['Negative','Positive'],
                 cmap='Blues')
ax.set(xlabel="Predicted Label", ylabel="True Label")
ax.set(title="mBERT");

In [ ]:
interaction_matrix, mbert_error_samples = check_error_from_predictions(mbert_pred_labels)

In [ ]:
mbert_mcnemar_result = do_mcnemar_test(mbert_pred_labels, 'mBERT')

In [ ]:
mbert_mcnemar_result

In [ ]:
interaction_matrix

In [ ]:
mbert_error_samples

In [ ]:
mbert_error_samples.to_excel('mbert_error_samples.xlsx', index=False)

In [ ]:
mbert_mcnemar_result

In [ ]:
run_check__bert_attention(mbert_error_samples, "aplikasi", trained_mbert_model, tokenizer)

In [ ]:
run_check__bert_attention(mbert_error_samples, "dari", trained_mbert_model, tokenizer)

#XLM-ROBERTA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

In [ ]:
xlmroberta_train_dataset = df_to_dataloader(X_train_statements, y_train, tokenizer, is_train=True)
xlmroberta_test_dataset = df_to_dataloader(X_test_statements, y_test, tokenizer, is_train=False)

In [ ]:
# CREATE XLMRoberta Model
classifier = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base", # model name
    num_labels = 2, # The number of output labels--2 for binary classification.
    output_attentions = True, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)

In [ ]:
trained_xlmroberta_model = train_model(classifier, xlmroberta_train_dataset)

In [ ]:
true_labels, xlmroberta_pred_labels = evaluate_model(trained_xlmroberta_model, xlmroberta_test_dataset)

In [ ]:
ax = sns.heatmap(confusion_matrix(true_labels, xlmroberta_pred_labels), annot=True, fmt='.0f',
                 xticklabels=['Negative', 'Positive'], yticklabels=['Negative','Positive'],
                 cmap='Blues')
ax.set(xlabel="Predicted Label", ylabel="True Label")
ax.set(title="XLM-RoBERTa");

In [ ]:
check_error_from_predictions(xlmroberta_pred_labels)

In [ ]:
roberta_mcnemar_result = do_mcnemar_test(xlmroberta_pred_labels, 'XLM-RoBERTa')

In [ ]:
roberta_mcnemar_result

#IndoBERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('indolem/indobert-base-uncased')

In [ ]:
indobert_train_dataset = df_to_dataloader(X_train_statements, y_train, tokenizer, is_train=True)
indobert_test_dataset = df_to_dataloader(X_test_statements, y_test, tokenizer, is_train=False)

In [ ]:
# CREATE IndoBERT Model
classifier = AutoModelForSequenceClassification.from_pretrained(
    "indolem/indobert-base-uncased", # model name
    num_labels = 2, # The number of output labels--2 for binary classification.
    output_attentions = True, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)

In [ ]:
trained_indobert_model = train_model(classifier, indobert_train_dataset)

In [ ]:
true_labels, indobert_pred_labels = evaluate_model(trained_indobert_model, indobert_test_dataset)

In [ ]:
ax = sns.heatmap(confusion_matrix(true_labels, indobert_pred_labels), annot=True, fmt='.0f',
                 xticklabels=['Negative', 'Positive'], yticklabels=['Negative','Positive'],
                 cmap='Blues')
ax.set(xlabel="Predicted Label", ylabel="True Label")
ax.set(title="IndoBERT");

In [ ]:
interaction_matrix, indobert_error_samples = check_error_from_predictions(indobert_pred_labels)

In [ ]:
indobert_mcnemar_result = do_mcnemar_test(indobert_pred_labels, 'IndoBERT')

In [ ]:
indobert_mcnemar_result

In [ ]:
interaction_matrix

In [ ]:
indobert_error_samples

In [ ]:
indobert_error_samples.to_excel('indobert_error_samples.xlsx', index=False)

In [ ]:
indobert_mcnemar_result

In [ ]:
run_check__bert_attention(indobert_error_samples, 'aplikasi', trained_indobert_model, tokenizer)

#OTHERS

In [ ]:
# training_set = X_train.copy().reset_index(drop=True)
# training_set['labels'] = y_train.reset_index(drop=True)
# training_set.to_excel("training_set.xlsx", index=False)

In [ ]:
# test_set = X_test.copy().reset_index(drop=True)
# test_set['labels'] = y_test.reset_index(drop=True)
# test_set.to_excel("test_set.xlsx", index=False)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

# 1. Kumpulkan prediksi enam model (urutan baris = urutan X_test)
pred_sources = {
    'pred_logistic_regression': lr_preds,
    'pred_linear_svc':          svm_preds,
    'pred_bert':                bert_pred_labels,
    'pred_mbert':               mbert_pred_labels,
    'pred_xlm_roberta':         xlmroberta_pred_labels,
    'pred_indobert':            indobert_pred_labels,
}

out = X_test.copy().reset_index(drop=True)          # reviewId, content, position, language, is_original
out['true_label'] = y_test.reset_index(drop=True).astype(int)
out['true_label_name'] = out['true_label'].map({0: 'Negative', 1: 'Positive'})

for col, preds in pred_sources.items():
    preds = np.asarray(preds).astype(int)
    assert len(preds) == len(out), f"{col}: {len(preds)} prediksi untuk {len(out)} baris"
    out[col] = preds

# 2. Urutkan agar mudah dibaca dan pastikan setiap varian unik
out = out.sort_values(['reviewId', 'language', 'position']).reset_index(drop=True)
assert not out.duplicated(['reviewId', 'language', 'position']).any(), "Ada varian ganda"

cols = ['reviewId', 'position', 'language', 'is_original', 'content',
        'true_label', 'true_label_name'] + list(pred_sources)
out = out[cols]

# 3. Cek silang dengan angka di paper (akurasi pada split yang dilaporkan)
print(f"{len(out)} baris, {out['reviewId'].nunique()} review")
for col in pred_sources:
    print(f"{col:28s} accuracy = {accuracy_score(out['true_label'], out[col]):.4f}")
# Harus cocok: LR 0.92, SVC 0.93, BERT 0.9364, mBERT 0.9758, XLM-R 1.0000, IndoBERT 0.9545

# 4. Simpan
out.to_csv('predictions_per_variant_test_split.csv', index=False, encoding='utf-8')

# 5. (Opsional) penugasan train/test per review agar splitnya bisa direproduksi
split_assign = X[['reviewId']].copy()
split_assign['split'] = 'train'
split_assign.iloc[test_idx, split_assign.columns.get_loc('split')] = 'test'
split_assign = split_assign.drop_duplicates('reviewId').sort_values('reviewId')
split_assign.to_csv('split_assignment_by_review.csv', index=False)
print(split_assign['split'].value_counts().to_dict())   # harus {'train': 217, 'test': 55}

# REPEATED RANDOM SPLITS — STABILITY ANALYSIS (Reviewer Point 1)

**Why this section exists:** the pre-review comment flagged that XLM-RoBERTa's zero-error
result on the single 80:20 split above needed to be checked against split variance before
being reported as evidence of robustness, rather than as a possible artifact of one lucky
(or unlucky, for the other models) train/test partition.

**What it does NOT do:** it does not replace the single-split run above. Tables 3–11 in the
paper — including the specific misclassified-review examples (e.g. *"Cukup membingungkan
aplikasi ini awal awal y."*) and the BERT/mBERT/IndoBERT attention-ratio analysis — are all
tied to that one reported split (`GroupShuffleSplit(random_state=42)`). Re-running with a
different split would change which reviews land in the test set and could silently break
those specific qualitative examples. So the code above is left untouched.

**What this section does instead**, following the reviewer's steps a–d:

1. Fix the split ratio at 80:20 (same as above).
2. Repeat a **grouped + stratified** random split `N_SPLITS` times, each with a different
   random seed. "Grouped" means every one of the six variants of a review stays together in
   either train or test (same requirement as the original `GroupShuffleSplit`, otherwise PCR/PFR
   can't be computed for a group). "Stratified" adds a guarantee `GroupShuffleSplit` does not
   give on its own: that the Positive/Negative ratio stays close to the full-dataset ratio in
   every split, not just on average.
3. For every split, train and evaluate all six models from scratch (fresh TF-IDF fit / fresh
   pretrained weights) and record accuracy, macro F1, Prediction Consistency Rate (PCR), and
   Prediction Flip Rate (PFR) — using the exact same PCR/PFR definitions as `check_error_from_predictions`
   above (Section 2.7, Eq. 1–2 in the paper), so the numbers here are directly comparable to Table 6.
4. Aggregate across splits into mean ± standard deviation per model, per metric.

**Runtime warning:** this retrains all four transformer models `N_SPLITS` times each on a T4
GPU. `N_SPLITS = 5` is the default below to keep this practical; the reviewer's alternative of
`N_SPLITS = 10` roughly doubles the runtime of this whole section. Change `N_SPLITS` in the
config cell if you want more iterations — nothing else needs to change.

### Sanity check: every review's 6 variants must share one label

The stratified split below stratifies at the *review* level (one label per `reviewId`), not the row level. If any review's variants disagree on label — which should not happen given the annotation guideline in Section 2.1 — stratification would silently use whichever label happens to be first. This check fails loudly instead.

In [ ]:
df = pd.read_csv('final_accepted_dataset - Accepted Variants.csv')

In [ ]:
# Clean and convert labels to numeric (int32 or int64)
df['labels'] = df['labels'].str.replace('POSITIF', '1').str.replace('NEGATIF', '0')
df['labels'] = pd.to_numeric(df['labels'])

#Shuffle first to simulate real data
df = shuffle(df, random_state=RANDOM_SEED).reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
X = df[['content','position','language','reviewId','is_original']]
y = df['labels']

In [ ]:
#create group for groupshufflesplit
groups = X.reviewId.reset_index(drop=True)

In [ ]:
_label_nunique = df.groupby('reviewId')['labels'].nunique()
_bad_groups = _label_nunique[_label_nunique > 1]
if len(_bad_groups) > 0:
    raise ValueError(
        f"{len(_bad_groups)} reviewId group(s) have inconsistent labels across their "
        f"variants -- fix the source data before running the stratified splits below. "
        f"Offending reviewId(s): {list(_bad_groups.index)}"
    )
else:
    print(f"OK: all {df['reviewId'].nunique()} review groups have a single consistent label.")

### Grouped + stratified split helper

In [ ]:
def grouped_stratified_split(X, y, groups, test_size=0.2, random_state=42):
    """
    Group-aware, label-stratified train/test split.

    sklearn's GroupShuffleSplit (used in the single-split run above) keeps all
    variants of a review together but does not guarantee the Positive/Negative
    ratio is preserved in every draw -- it only holds on average across many
    draws. For a single reported split that was an acceptable simplification;
    for a repeated-splits stability check it's worth removing that source of
    noise so the variance we observe reflects model/data instability rather
    than incidental class-imbalance differences between iterations.

    Approach: collapse to one row per reviewId (using each review's single,
    already-verified label), run a stratified train_test_split on the *groups*,
    then expand the chosen groups back out to every row (all 6 variants) that
    belongs to them.
    """
    group_labels = (
        pd.DataFrame({'reviewId': groups.values, 'labels': y.values})
        .drop_duplicates('reviewId')
        .set_index('reviewId')['labels']
    )

    train_groups, test_groups = train_test_split(
        group_labels.index,
        test_size=test_size,
        random_state=random_state,
        stratify=group_labels.values,
    )

    train_groups = set(train_groups)
    test_groups = set(test_groups)

    train_idx = np.where(groups.isin(train_groups).values)[0]
    test_idx = np.where(groups.isin(test_groups).values)[0]

    # same leakage check as the single-split run above
    overlap = train_groups & test_groups
    assert not overlap, f"Leakage detected: {len(overlap)} reviewId(s) in both splits"

    return train_idx, test_idx

### Silent metrics helper (accuracy, macro F1, PCR, PFR)

Same PCR/PFR algorithm as `check_error_from_predictions` above, but returns numbers instead of printing tables -- this is what gets called dozens of times inside the loop below.

In [ ]:
def compute_split_metrics(X_test, y_test, predictions):
    """
    Returns {'accuracy', 'macro_f1', 'pcr', 'pfr'} for one model on one split.
    Mirrors the exact consistency/flip-rate logic used in
    check_error_from_predictions() so results are directly comparable to Table 6.
    """
    results_df = X_test.copy().reset_index(drop=True)
    results_df['labels'] = pd.Series(y_test).reset_index(drop=True)
    results_df['predicted_label'] = predictions

    accuracy = accuracy_score(results_df['labels'], results_df['predicted_label'])
    macro_f1 = f1_score(results_df['labels'], results_df['predicted_label'], average='macro')

    total_groups = results_df['reviewId'].nunique()
    consistent_groups = 0
    total_flips = 0
    total_comparisons = 0

    for review_id, group in results_df.groupby('reviewId'):
        original_row = group[group['is_original'] == 'Y']
        if original_row.empty:
            # Should not happen -- grouped splitting keeps all 6 variants (including
            # the original) together. Skip defensively rather than crash a long run.
            continue
        scalar_label = original_row.index[0]
        positional_index = group.index.get_loc(scalar_label)
        preds = group['predicted_label'].values

        if len(set(preds)) == 1:
            consistent_groups += 1

        base_pred = preds[positional_index]
        for i, p in enumerate(preds):
            if i == positional_index:
                continue
            total_comparisons += 1
            if p != base_pred:
                total_flips += 1

    consistency_rate = consistent_groups / total_groups if total_groups else float('nan')
    flip_rate = total_flips / total_comparisons if total_comparisons else float('nan')

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'pcr': consistency_rate,
        'pfr': flip_rate,
    }

### Per-model training functions (one call = one split's worth of training)

In [ ]:
def run_lr_iteration(X_train_stmt, y_train, X_test_stmt, X_test, y_test):
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(preprocessor=preprocess_text, stop_words=None,
                                   ngram_range=(1, 2), max_df=0.95)),
        ('model', LogisticRegression(max_iter=1000, solver='lbfgs', C=1)),
    ])
    pipeline.fit(X_train_stmt, y_train)
    preds = pipeline.predict(X_test_stmt)
    return compute_split_metrics(X_test, y_test, preds)


def run_svm_iteration(X_train_stmt, y_train, X_test_stmt, X_test, y_test):
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(preprocessor=preprocess_text, stop_words=None,
                                   ngram_range=(1, 2), max_df=0.95)),
        ('model', LinearSVC(C=1, max_iter=1000, class_weight='balanced')),
    ])
    pipeline.fit(X_train_stmt, y_train)
    preds = pipeline.predict(X_test_stmt)
    return compute_split_metrics(X_test, y_test, preds)


def run_transformer_iteration(hf_model_name, X_train_stmt, y_train, X_test_stmt, X_test, y_test):
    tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
    train_loader = df_to_dataloader(X_train_stmt, y_train, tokenizer, is_train=True)
    test_loader = df_to_dataloader(X_test_stmt, y_test, tokenizer, is_train=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        hf_model_name,
        num_labels=2,
        output_attentions=False,   # not needed for this stability check -- saves memory
        output_hidden_states=False,
    )
    trained_model = train_model(model, train_loader)
    _, preds = evaluate_model(trained_model, test_loader)

    metrics = compute_split_metrics(X_test, y_test, preds)

    # free GPU memory before the next model / iteration -- this loop trains up to
    # 4 x N_SPLITS transformer models back to back on a single GPU
    del trained_model, model, train_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

### Config

Change `N_SPLITS` here if you want the reviewer's alternative of 10 iterations instead of 5. Everything below adapts automatically.

In [ ]:
N_SPLITS = 5  # reviewer suggested N=5 or N=10
SPLIT_SEEDS = [42 + i * 111 for i in range(N_SPLITS)]

TRANSFORMER_MODELS = {
    'BERT': 'bert-base-uncased',
    'mBERT': 'bert-base-multilingual-uncased',
    'XLM-RoBERTa': 'xlm-roberta-base',
    'IndoBERT': 'indolem/indobert-base-uncased',
}

print(f"N_SPLITS = {N_SPLITS}")
print(f"SPLIT_SEEDS = {SPLIT_SEEDS}")

### Main loop: retrain all 6 models on each of the N splits

This is the expensive cell -- it trains Logistic Regression, Linear SVC, and all four transformer models, once per split.

In [ ]:
repeated_split_results = []

for split_i, seed in enumerate(SPLIT_SEEDS):
    print(f"\n{'=' * 70}\nSPLIT {split_i + 1}/{N_SPLITS}  (seed={seed})\n{'=' * 70}")
    set_reproducibility_seed(seed)

    train_idx, test_idx = grouped_stratified_split(X, y, groups, test_size=0.2, random_state=seed)
    X_train_i, X_test_i = X.iloc[train_idx], X.iloc[test_idx]
    y_train_i, y_test_i = y.iloc[train_idx], y.iloc[test_idx]
    X_train_stmt_i = X_train_i['content']
    X_test_stmt_i = X_test_i['content']

    print(f"  train={len(X_train_i)} rows ({X_train_i.reviewId.nunique()} reviews) | "
          f"test={len(X_test_i)} rows ({X_test_i.reviewId.nunique()} reviews)")
    print(f"  train label balance: {y_train_i.value_counts(normalize=True).round(3).to_dict()}")
    print(f"  test  label balance: {y_test_i.value_counts(normalize=True).round(3).to_dict()}")

    print("  -> Logistic Regression")
    m = run_lr_iteration(X_train_stmt_i, y_train_i, X_test_stmt_i, X_test_i, y_test_i)
    repeated_split_results.append({'model': 'Logistic Regression', 'split': split_i, 'seed': seed, **m})

    print("  -> Linear SVC")
    m = run_svm_iteration(X_train_stmt_i, y_train_i, X_test_stmt_i, X_test_i, y_test_i)
    repeated_split_results.append({'model': 'Linear SVC', 'split': split_i, 'seed': seed, **m})

    for model_label, hf_name in TRANSFORMER_MODELS.items():
        print(f"  -> {model_label}")
        m = run_transformer_iteration(hf_name, X_train_stmt_i, y_train_i, X_test_stmt_i, X_test_i, y_test_i)
        repeated_split_results.append({'model': model_label, 'split': split_i, 'seed': seed, **m})

print("\nAll splits done.")

### Raw per-split results

In [ ]:
repeated_split_df = pd.DataFrame(repeated_split_results)
repeated_split_df

### Aggregated summary — mean ± std per model

This is the table to drop into the paper (e.g. as Table 3b, or as new columns next to Table 3/Table 6) to answer reviewer point 1 directly.

In [ ]:
MODEL_ORDER = ['Logistic Regression', 'Linear SVC', 'BERT', 'mBERT', 'XLM-RoBERTa', 'IndoBERT']

summary = repeated_split_df.groupby('model')[['accuracy', 'macro_f1', 'pcr', 'pfr']].agg(['mean', 'std'])
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index()
summary['model'] = pd.Categorical(summary['model'], categories=MODEL_ORDER, ordered=True)
summary = summary.sort_values('model').reset_index(drop=True)

# round for display / paper table -- raw CSV export below keeps full precision
summary_display = summary.copy()
for c in summary_display.columns:
    if c != 'model':
        summary_display[c] = summary_display[c].astype(float).round(4)

summary_display

### Export results

In [ ]:
repeated_split_df.to_csv('repeated_split_raw_results.csv', index=False)
summary.to_csv('repeated_split_summary_mean_std.csv', index=False)
print("Saved: repeated_split_raw_results.csv (per-split, per-model rows)")
print("Saved: repeated_split_summary_mean_std.csv (mean/std per model, ready for the paper table)")

### Reading these results for the paper

- If **XLM-RoBERTa's mean accuracy/F1 stays at or near 1.00 with a small std** across all
  `N_SPLITS` runs, that supports reading its original zero-error result as genuine stability
  rather than a one-off artifact of the reported split -- strengthens the claim in Section 3.2.
- If XLM-RoBERTa's std is **not** small, or another split produces non-zero errors, that is
  itself the answer to the open question raised in the Discussion ("the test set is too small to
  expose a shortcut it still relies on") -- report the mean ± std honestly and revise that
  paragraph (originally around the "XLM-RoBERTa's zero-error result is the one finding this
  shortcut argument cannot explain" discussion) to reflect what the repeated splits actually show.
- Compare `pcr`/`pfr` std across models the same way: a model whose PCR/PFR barely moves across
  splits (e.g. Linear SVC, which was already near-ceiling on PCR) is behaving predictably;
  large swings in IndoBERT's or mBERT's PCR/PFR across splits would mean the single-split numbers
  in Table 6 should be reported as one draw from a wider distribution, not as the model's fixed
  behavior.
- `repeated_split_raw_results.csv` has one row per (model, split) if you want to run your own
  significance test (e.g. paired t-test across splits) on top of the mean/std reported here.

# CAUSAL EVIDENCE FOR THE SHORTCUT CLAIMS (Reviewer A, point C1)

**Why this section exists.** The review pointed out that the shortcut evidence in the paper is correlational: the 2.62-4.03x attention ratio comes from a handful of misclassified sentences with no control, attention is not a causal explanation, and the link between trigger tokens ("aplikasi", "dari", "kalau", "travel") and sentiment in the training data was never shown. XLM-RoBERTa was also never analysed because it made no errors. This section adds four checks, each aimed at one part of that criticism.

| Part | Question it answers | Needs GPU |
|---|---|---|
| 1. Token statistics + Logistic Regression / Linear SVC coefficients | Are the trigger tokens actually associated with a class in the training data? How common are they overall? | no |
| 2. Attention controls | Is the attention ratio specific to misclassified sentences, or just as high on correctly classified ones? How many instances are behind it? How does attention to the trigger token compare with attention to the sentiment cue itself? | yes (light) |
| 3. Intervention tests (ablation / replacement) | If the trigger token is removed or replaced, does the prediction change more than when a random word is removed? Does this also hold for XLM-RoBERTa? | yes (light) |
| 4. Integrated Gradients | Does gradient-based attribution agree with the attention picture? | yes (about 10 min on a T4 with the defaults) |

**What this section does NOT do.** It does not retrain anything and it does not touch the earlier cells. Everything uses the models and the 80:20 `GroupShuffleSplit(random_state=42)` test set from the single-split run, because that run is the one behind Tables 3-11 and the attention analysis in the paper.

**How to run it.** Run cells 0-121 first so the trained models (`trained_bert_model`, `trained_mbert_model`, `trained_xlmroberta_model`, `trained_indobert_model`), `lr_pipeline` and `svm_pipeline` exist in memory. The repeated-split section (cells 126-151) is *not* required and can be skipped. All new variables start with `cz_` / `CZ_` so they cannot overwrite anything above.

**Report whatever comes out.** These checks can strengthen the shortcut claims, weaken them, or show they hold for some models and not others. Any of those outcomes is a valid result for the paper; the text should be written from the numbers, not the other way round.

### Setup and state check

In [ ]:
import sys, subprocess

try:
    import captum
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'captum'])
    import captum
from captum.attr import LayerIntegratedGradients
from scipy.stats import fisher_exact

_REQUIRED = [
    'X_train', 'y_train', 'X_test', 'y_test', 'df',
    'lr_pipeline', 'svm_pipeline',
    'trained_bert_model', 'trained_mbert_model', 'trained_xlmroberta_model', 'trained_indobert_model',
    'lr_preds', 'svm_preds', 'bert_pred_labels', 'mbert_pred_labels',
    'xlmroberta_pred_labels', 'indobert_pred_labels',
]
_missing = [n for n in _REQUIRED if n not in globals()]
if _missing:
    raise RuntimeError(
        "Missing from the notebook state: " + ", ".join(_missing) +
        ". Run cells 0-121 first (data split, Logistic Regression, Linear SVC and the four transformer models)."
    )
print("State check OK. Test set:", len(X_test), "rows /", X_test.reviewId.nunique(), "reviews")

### Configuration

`CZ_TARGET_TOKENS` are the background tokens named in the paper's error analysis. `CZ_PRIMARY_TOKEN` is the one used for the attention and Integrated Gradients controls. `CZ_FILLERS` are the neutral words used when a token is *replaced* instead of deleted. Change them here and everything below follows.

In [ ]:
CZ_SEED = 0
CZ_TARGET_TOKENS = ['aplikasi', 'dari', 'kalau', 'cari', 'semua', 'travel']
CZ_PRIMARY_TOKEN = 'aplikasi'
CZ_FILLERS = ['hal', 'proses', 'layanan']   # neutral replacement words (Part 3)
CZ_N_RANDOM_DRAWS = 5                        # random-word deletions per row, used as the control (Part 3)
CZ_BOOT = 2000                               # cluster-bootstrap resamples for confidence intervals
CZ_IG_STEPS = 32                             # Integrated Gradients interpolation steps (Part 4)
CZ_RUN_IG = True                             # set False to skip Part 4
CZ_MAX_LEN = MAX_SEQ_LEN                     # same truncation length as training (60)

CZ_ALL_MODELS = ['Logistic Regression', 'Linear SVC', 'BERT', 'mBERT', 'XLM-RoBERTa', 'IndoBERT']

CZ_TRANSFORMERS = {
    'BERT': trained_bert_model,
    'mBERT': trained_mbert_model,
    'XLM-RoBERTa': trained_xlmroberta_model,
    'IndoBERT': trained_indobert_model,
}
CZ_HF_NAMES = {
    'BERT': 'bert-base-uncased',
    'mBERT': 'bert-base-multilingual-uncased',
    'XLM-RoBERTa': 'xlm-roberta-base',
    'IndoBERT': 'indolem/indobert-base-uncased',
}
# the global `tokenizer` was overwritten while training the models above, so load each one again
CZ_TOK = {name: AutoTokenizer.from_pretrained(hf) for name, hf in CZ_HF_NAMES.items()}
for _name, _tok in CZ_TOK.items():
    if not _tok.is_fast:
        raise RuntimeError(f"{_name}: a fast tokenizer is required (character offsets are used to locate tokens).")

for _m in CZ_TRANSFORMERS.values():
    _m.to(device)
    _m.eval()

# predictions stored by the original single-split run, in X_test order
CZ_STORED_PREDS = {
    'Logistic Regression': np.asarray(lr_preds),
    'Linear SVC': np.asarray(svm_preds),
    'BERT': np.asarray(bert_pred_labels),
    'mBERT': np.asarray(mbert_pred_labels),
    'XLM-RoBERTa': np.asarray(xlmroberta_pred_labels),
    'IndoBERT': np.asarray(indobert_pred_labels),
}
print("Configuration loaded.")

### Test-set table with the sentiment-cue location

The dataset table `df` has a `cue` column (the Indonesian sentiment expression of each review). For **Indonesian** rows the cue is located by exact string match. For **English** rows the translated cue is not stored, so it is recovered as the words that do not appear in any Indonesian variant of the same review (1-4 words, otherwise the cue is left unknown). Coverage is printed below; rows without a located cue are simply left out of the cue-specific comparisons.

In [ ]:
cz_test = X_test.copy().reset_index(drop=True)
cz_test['label'] = y_test.reset_index(drop=True).astype(int)

_meta = df[['reviewId', 'content', 'position', 'language', 'cue']].drop_duplicates(
    ['reviewId', 'content', 'position', 'language'])
cz_test = cz_test.merge(_meta, on=['reviewId', 'content', 'position', 'language'], how='left')
assert len(cz_test) == len(X_test), "merge with the cue column changed the number of test rows"
_n_nocue = int(cz_test['cue'].isna().sum())
if _n_nocue:
    print(f"WARNING: {_n_nocue} test rows could not be matched to a cue in `df`. The dataset table `df` probably differs "
          f"from the CSV used for the original split (cell 22 vs cell 128). Cue-specific results will skip those rows.")

_word_re = re.compile(r"[A-Za-z0-9']+")

def cz_word_spans(text):
    return [(m.group().lower(), m.start(), m.end()) for m in _word_re.finditer(text)]

_indo_words = {
    rid: {w for t in g['content'] for w, _, _ in cz_word_spans(t)}
    for rid, g in cz_test[cz_test['language'] == 'Indonesian'].groupby('reviewId')
}

def cz_find_cue_spans(row):
    text = row['content']
    cue = str(row['cue']).strip().lower() if pd.notna(row['cue']) else ''
    if row['language'] == 'Indonesian':
        k = text.lower().find(cue) if cue else -1
        if k >= 0:
            return [(k, k + len(cue))], 'indonesian_exact_match'
        return None, 'not_found'
    novel = [(s, e) for w, s, e in cz_word_spans(text) if w not in _indo_words.get(row['reviewId'], set())]
    if 1 <= len(novel) <= 4:
        return novel, 'english_word_diff'
    return None, 'not_found'

_found = [cz_find_cue_spans(r) for _, r in cz_test.iterrows()]
cz_test['cue_spans'] = pd.Series([f[0] for f in _found], index=cz_test.index, dtype=object)
cz_test['cue_source'] = [f[1] for f in _found]

print(cz_test.groupby(['language', 'cue_source']).size().unstack(fill_value=0))
print("\nSpot check (cue text recovered from the spans):")
for _, r in cz_test[cz_test['cue_spans'].notna()].sample(8, random_state=1).iterrows():
    print(f"  [{r['language'][:3]}] {r['content']!r}  ->  {[r['content'][s:e] for s, e in r['cue_spans']]}")

## Part 1 - Are the trigger tokens actually tied to a class in the training data?

Counts are made at the **review** level (a review counts once even though it has six variants), because the six variants share the same background words and are not independent observations. `pct_positive_given_token` should be compared with `base_rate_positive` (the share of positive reviews in training), and `pct_of_train_reviews` shows how common the token is overall, which matters for judging whether high attention to it is surprising.

In [ ]:
def cz_target_spans(text, token):
    # same matching rule as the attention analysis in the paper: any word containing the token
    # (so "aplikasi" also matches "aplikasinya")
    return [(m.start(), m.end()) for m in re.finditer(r"\w*" + re.escape(token) + r"\w*", text, flags=re.IGNORECASE)]

def cz_has_token(text, token):
    return len(cz_target_spans(text, token)) > 0

_tr = X_train[['reviewId', 'content']].copy()
_tr['label'] = np.asarray(y_train).astype(int)
_te = X_test[['reviewId', 'content']].copy()
_te['label'] = np.asarray(y_test).astype(int)

def _review_level(frame, token):
    f = frame.copy()
    f['has'] = f['content'].map(lambda t: cz_has_token(t, token))
    return f.groupby('reviewId').agg(has=('has', 'max'), label=('label', 'first'))

_base_rate = _review_level(_tr, CZ_TARGET_TOKENS[0])['label'].mean()
_rows = []
for tok in CZ_TARGET_TOKENS:
    g = _review_level(_tr, tok)
    a = int(((g['has']) & (g['label'] == 1)).sum())   # token present, positive
    b = int(((g['has']) & (g['label'] == 0)).sum())   # token present, negative
    c = int(((~g['has']) & (g['label'] == 1)).sum())
    d = int(((~g['has']) & (g['label'] == 0)).sum())
    odds, p = fisher_exact([[a, b], [c, d]]) if (a + b) > 0 else (np.nan, np.nan)
    gt = _review_level(_te, tok)
    _rows.append({
        'token': tok,
        'train_reviews_with_token': a + b,
        'pct_of_train_reviews': round(100 * (a + b) / len(g), 1),
        'positive_reviews': a, 'negative_reviews': b,
        'pct_positive_given_token': round(100 * a / (a + b), 1) if (a + b) else np.nan,
        'base_rate_positive': round(100 * _base_rate, 1),
        'odds_ratio_positive': round(odds, 2) if np.isfinite(odds) else odds,
        'fisher_p': p,
        'test_reviews_with_token': int(gt['has'].sum()),
    })
cz_token_stats = pd.DataFrame(_rows)
print(f"Training reviews: {_review_level(_tr, CZ_TARGET_TOKENS[0]).shape[0]}  (positive base rate {100 * _base_rate:.1f}%)")
print("Rows with fewer than ~10 reviews carry little statistical weight; read their p-values accordingly.\n")
cz_token_stats

### Logistic Regression and Linear SVC coefficients for the same tokens

A positive coefficient pushes towards **Positive**. `rank_from_most_positive` is the rank among all TF-IDF features (1 = most positive), so a token near rank 1 or near the last rank is one the model relies on heavily. Only the exact unigram is shown (e.g. "aplikasi" but not "aplikasinya").

In [ ]:
def cz_coef_table(pipe, model_name):
    vec = pipe.named_steps['tfidf']
    clf = pipe.named_steps['model']
    feats = np.array(vec.get_feature_names_out())
    coef = clf.coef_[0]
    rank = pd.Series(coef, index=feats).rank(ascending=False, method='min')
    out = []
    for tok in CZ_TARGET_TOKENS:
        if tok in rank.index:
            out.append({'model': model_name, 'token': tok, 'coefficient': round(float(pd.Series(coef, index=feats)[tok]), 3),
                        'rank_from_most_positive': int(rank[tok]), 'n_features': len(feats)})
        else:
            out.append({'model': model_name, 'token': tok, 'coefficient': np.nan,
                        'rank_from_most_positive': np.nan, 'n_features': len(feats)})
    top = pd.Series(coef, index=feats).sort_values()
    return pd.DataFrame(out), top

_lr_t, _lr_top = cz_coef_table(lr_pipeline, 'Logistic Regression')
_svm_t, _svm_top = cz_coef_table(svm_pipeline, 'Linear SVC')
cz_coef_df = pd.concat([_lr_t, _svm_t], ignore_index=True)
display(cz_coef_df)

print("\nLogistic Regression: 12 most negative and 12 most positive features")
print(pd.concat([_lr_top.head(12), _lr_top.tail(12)]).round(3).to_string())

### Shared helpers (bootstrap confidence intervals, token location)

All confidence intervals below come from a **cluster bootstrap over `reviewId`**: whole reviews (with all their variants) are resampled, because the six variants of a review are not independent observations. A plain row-level interval would be too narrow.

In [ ]:
def cz_boot_mean_ci(frame, value_col, cluster_col='reviewId', B=None, seed=None):
    """Mean of `value_col` with a cluster-bootstrap 95% CI. Returns (mean, lo, hi)."""
    B = CZ_BOOT if B is None else B
    seed = CZ_SEED if seed is None else seed
    d = frame[[cluster_col, value_col]].dropna()
    if d.empty:
        return np.nan, np.nan, np.nan
    g = d.groupby(cluster_col)[value_col].agg(['sum', 'count'])
    sums, counts = g['sum'].values.astype(float), g['count'].values.astype(float)
    k = len(sums)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, k, size=(B, k))
    means = sums[idx].sum(axis=1) / counts[idx].sum(axis=1)
    return float(d[value_col].mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def cz_summarize(frame, group_cols, value_cols):
    """Tidy table: mean, median, cluster-bootstrap CI, n rows and n reviews per group and metric."""
    out = []
    for keys, g in frame.groupby(group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        for v in value_cols:
            valid = g[g[v].notna()]
            mean, lo, hi = cz_boot_mean_ci(g, v)
            out.append({**dict(zip(group_cols, keys)), 'metric': v, 'n_rows': len(valid),
                        'n_reviews': valid['reviewId'].nunique(), 'mean': mean, 'median': valid[v].median(),
                        'ci_lo': lo, 'ci_hi': hi})
    return pd.DataFrame(out)


def cz_token_idx(offsets, spans):
    """Indices of tokens whose character range overlaps any (start, end) span. Special tokens have (0, 0)."""
    if not spans:
        return []
    idx = []
    for i, (s, e) in enumerate(offsets):
        if e <= s:
            continue
        if any(s < se and e > ss for ss, se in spans):
            idx.append(i)
    return idx

print("Helpers defined.")

## Part 2 - Attention controls

The earlier analysis reported `attention to the trigger token / mean attention to the other tokens` for misclassified sentences only. This part computes **exactly the same ratio** (same definition: layer-and-head averaged, all query positions) for *every* test sentence that contains the trigger token, split into correctly and incorrectly classified, and adds three things that were missing:

- the **number of instances and reviews** behind each figure;
- the same ratio for the **sentiment cue** in the same sentence, so "the trigger token versus the cue" is measured directly (the earlier ratio compared the trigger with the *average other token*, not with the cue);
- a **per-piece ratio** (ratio divided by the number of word pieces), because a word split into several pieces adds up their attention while the baseline is a per-token mean, which inflates the plain ratio for multi-piece words;
- the **[CLS] row only** (`*_cls`), since the classifier reads the [CLS] representation.

XLM-RoBERTa is included. It has no errors, so its error group is empty, but its correct-sentence ratios are still informative.

In [ ]:
def cz_force_eager(model):
    # sdpa / flash attention backends do not return attention weights; eager does
    try:
        model.set_attn_implementation('eager')
    except Exception:
        model.config._attn_implementation = 'eager'


@torch.no_grad()
def cz_attention_row(name, text, target_spans, cue_spans):
    model, tok = CZ_TRANSFORMERS[name], CZ_TOK[name]
    enc = tok(text, return_tensors='pt', return_offsets_mapping=True, truncation=True, max_length=CZ_MAX_LEN)
    offsets = enc.pop('offset_mapping')[0].tolist()
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc, output_attentions=True)
    if out.attentions is None or out.attentions[0] is None:
        cz_force_eager(model)
        out = model(**enc, output_attentions=True)
        if out.attentions is None or out.attentions[0] is None:
            raise RuntimeError(f"{name}: the model does not return attention weights even with the eager implementation.")
    A = torch.stack(out.attentions)[:, 0].mean(dim=1)          # [layers, seq, seq], head-averaged
    ids = enc['input_ids'][0].tolist()
    skip = {tok.cls_token_id, tok.sep_token_id, tok.pad_token_id}   # same exclusions as the original analysis
    valid = [i for i, t in enumerate(ids) if t not in skip]

    def ratios(T):
        if not T:
            return np.nan, np.nan, np.nan
        Tset = set(T)
        O = [i for i in valid if i not in Tset]
        if not O:
            return np.nan, np.nan, np.nan
        recv = A[:, :, T].sum(dim=-1).mean(dim=-1)              # attention mass received by T, per layer
        base = A[:, :, O].mean(dim=(1, 2))                      # mean attention received by an other token
        r = (recv.mean() / base.mean().clamp_min(1e-8)).item()
        rc = (A[:, 0, T].sum(dim=-1).mean() / A[:, 0, O].mean(dim=-1).mean().clamp_min(1e-8)).item()
        return r, r / len(T), rc

    T = cz_token_idx(offsets, target_spans)
    C = cz_token_idx(offsets, cue_spans)
    tr, trp, trc = ratios(T)
    cr, crp, crc = ratios(C)
    return {'target_pieces': len(T), 'cue_pieces': len(C),
            'target_ratio': tr, 'target_ratio_per_piece': trp, 'target_ratio_cls': trc,
            'cue_ratio': cr, 'cue_ratio_per_piece': crp, 'cue_ratio_cls': crc}


_records = []
for name in CZ_TRANSFORMERS:
    preds = CZ_STORED_PREDS[name]
    for i, r in cz_test.iterrows():
        tsp = cz_target_spans(r['content'], CZ_PRIMARY_TOKEN)
        csp = r['cue_spans'] if isinstance(r['cue_spans'], list) else []
        if not tsp and not csp:
            continue
        res = cz_attention_row(name, r['content'], tsp, csp)
        res.update({'model': name, 'row': i, 'reviewId': r['reviewId'], 'position': r['position'],
                    'language': r['language'], 'label': int(r['label']), 'pred': int(preds[i]),
                    'outcome': 'correct' if preds[i] == r['label'] else 'error',
                    'has_target': bool(tsp), 'has_cue': bool(csp)})
        _records.append(res)

cz_att = pd.DataFrame(_records)
cz_att['target_over_cue'] = cz_att['target_ratio'] / cz_att['cue_ratio']
print("Attention computed for", len(cz_att), "(model, sentence) pairs.")

#### Sanity check: does this reproduce the ratios reported in the paper?

The BERT misclassified sentences containing "aplikasi" are the 2.62-4.03 range in the abstract. If the numbers below do not match that range, stop and check the cell above before trusting anything else in this part.

In [ ]:
_chk = cz_att[(cz_att['model'] == 'BERT') & (cz_att['outcome'] == 'error') & (cz_att['has_target'])]
print(f"BERT, misclassified, contains '{CZ_PRIMARY_TOKEN}':  {len(_chk)} sentences from {_chk['reviewId'].nunique()} distinct reviews")
if len(_chk):
    print(f"  head-averaged ratio range: {_chk['target_ratio'].min():.2f}x - {_chk['target_ratio'].max():.2f}x   (paper: 2.62x - 4.03x)")
else:
    print("  no such sentences found: check that trained_bert_model / bert_pred_labels come from the original split run.")
print("\nThe instance count above is the number to report next to the ratio range. Note how few distinct reviews stand behind it.")

#### Results: trigger token versus sentiment cue, correct versus misclassified sentences

How to read the table: `metric` is the ratio listed; `n_rows` / `n_reviews` are the instances behind the mean; the interval is the cluster-bootstrap 95% CI. If `aplikasi` attention is just as high in the `correct` rows as in the `error` rows, high attention to it cannot explain the errors, because it is simply a very common token that every sentence contains. `target_over_cue` greater than 1 means more attention on the trigger token than on the cue in the same sentence.

In [ ]:
_metrics = ['target_ratio', 'target_ratio_per_piece', 'target_ratio_cls',
            'cue_ratio', 'cue_ratio_per_piece', 'cue_ratio_cls', 'target_over_cue']
cz_att_summary = cz_summarize(cz_att, ['model', 'outcome'], _metrics)
cz_att_summary[['mean', 'median', 'ci_lo', 'ci_hi']] = cz_att_summary[['mean', 'median', 'ci_lo', 'ci_hi']].round(2)

print("Head-averaged ratio of attention to the trigger token / to the average other token (the paper's definition)")
display(cz_att_summary[cz_att_summary['metric'] == 'target_ratio'].drop(columns='metric').reset_index(drop=True))
print("Same, for the sentiment cue")
display(cz_att_summary[cz_att_summary['metric'] == 'cue_ratio'].drop(columns='metric').reset_index(drop=True))
print("Trigger token versus cue, within the same sentence (>1 = more attention on the trigger token)")
display(cz_att_summary[cz_att_summary['metric'] == 'target_over_cue'].drop(columns='metric').reset_index(drop=True))
print("Per-piece ratio for the trigger token (corrects for words split into several pieces)")
display(cz_att_summary[cz_att_summary['metric'] == 'target_ratio_per_piece'].drop(columns='metric').reset_index(drop=True))
print("[CLS] row only, trigger token")
display(cz_att_summary[cz_att_summary['metric'] == 'target_ratio_cls'].drop(columns='metric').reset_index(drop=True))

## Part 3 - Intervention tests (ablation and replacement)

This is the closest thing to a causal test in this section. For every test sentence that contains a trigger token, the token is

1. **deleted**, or
2. **replaced** by a neutral word (each word in `CZ_FILLERS`, averaged), and the prediction is recomputed.

Two controls make the numbers interpretable:

- **Random-word deletion**: delete one randomly chosen *other* word (not the trigger token, not the sentiment cue when known), repeated `CZ_N_RANDOM_DRAWS` times per sentence. This is the baseline flip rate of "remove any word". A trigger token only looks special if deleting it flips predictions *more* than this.
- **Cue deletion** (positive control): delete the sentiment cue itself. A model that reads the cue should be affected by this.

`flip` means the predicted label changes. `repair` is the share of originally wrong predictions that become correct after the edit; `break` is the share of originally correct predictions that become wrong. All three TF-IDF/transformer models are treated the same way, using the sign of the positive-class margin (logit difference for transformers, `decision_function` for Logistic Regression and Linear SVC). XLM-RoBERTa is included, which answers the review's point that it was never analysed.

In [ ]:
@torch.no_grad()
def cz_margins_transformer(name, texts, batch_size=64):
    model, tok = CZ_TRANSFORMERS[name], CZ_TOK[name]
    model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(list(texts[i:i + batch_size]), add_special_tokens=True, max_length=CZ_MAX_LEN,
                  padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')
        logits = model(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device)).logits
        out.append((logits[:, 1] - logits[:, 0]).float().cpu().numpy())
    return np.concatenate(out)


def cz_margins(name, texts):
    """Positive-class margin; > 0 means the model predicts Positive."""
    texts = [str(t) for t in texts]
    if name == 'Logistic Regression':
        return np.asarray(lr_pipeline.decision_function(texts))
    if name == 'Linear SVC':
        return np.asarray(svm_pipeline.decision_function(texts))
    return cz_margins_transformer(name, texts)


# Sanity check: margins must reproduce the predictions stored by the original run
cz_orig_margin = {}
for name in CZ_ALL_MODELS:
    m = cz_margins(name, cz_test['content'])
    agree = float(((m > 0).astype(int) == CZ_STORED_PREDS[name]).mean())
    cz_orig_margin[name] = m
    flag = '' if agree >= 0.99 else '   <-- CHECK: should be ~1.0000'
    print(f"{name:20s} agreement with stored predictions: {agree:.4f}{flag}")

In [ ]:
def cz_clean(text):
    text = re.sub(r"\s+([,.;!?])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return re.sub(r"^[,.;!?\s]+", "", text)

def cz_merge_spans(spans):
    spans = sorted(spans)
    merged = []
    for s, e in spans:
        if merged and s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    return merged

def cz_delete_spans(text, spans):
    for s, e in reversed(cz_merge_spans(spans)):
        text = text[:s] + text[e:]
    return cz_clean(text)

def cz_replace_spans(text, spans, filler):
    for s, e in reversed(cz_merge_spans(spans)):
        text = text[:s] + filler + text[e:]
    return cz_clean(text)

def cz_random_delete(text, avoid_spans, rng):
    words = [(m.start(), m.end()) for m in re.finditer(r"[A-Za-z0-9']+", text)]
    cand = [(s, e) for s, e in words if not any(s < se and e > ss for ss, se in avoid_spans)]
    if not cand:
        return None
    s, e = cand[int(rng.integers(len(cand)))]
    return cz_clean(text[:s] + text[e:])


cz_rng = np.random.default_rng(CZ_SEED)
_recs = []
for i, r in cz_test.iterrows():
    cue_sp = r['cue_spans'] if isinstance(r['cue_spans'], list) else []
    for tok in CZ_TARGET_TOKENS:
        tsp = cz_target_spans(r['content'], tok)
        if not tsp:
            continue
        t = cz_delete_spans(r['content'], tsp)
        if t:
            _recs.append({'row': i, 'token': tok, 'kind': 'delete_token', 'draw': '0', 'text': t})
        for f in CZ_FILLERS:
            t = cz_replace_spans(r['content'], tsp, f)
            if t:
                _recs.append({'row': i, 'token': tok, 'kind': 'replace_token', 'draw': f, 'text': t})
        for d in range(CZ_N_RANDOM_DRAWS):
            t = cz_random_delete(r['content'], tsp + cue_sp, cz_rng)
            if t:
                _recs.append({'row': i, 'token': tok, 'kind': 'delete_random', 'draw': str(d), 'text': t})
    if cue_sp:
        t = cz_delete_spans(r['content'], cue_sp)
        if t:
            _recs.append({'row': i, 'token': '(cue)', 'kind': 'delete_cue', 'draw': '0', 'text': t})

cz_recs = pd.DataFrame(_recs)
print("Counterfactual texts:", len(cz_recs))
print(cz_recs.groupby(['token', 'kind']).size().unstack(fill_value=0))
print("\nExample edits:")
for _, r in cz_recs.sample(6, random_state=3).iterrows():
    print(f"  [{r['token']}/{r['kind']}]  {cz_test.loc[r['row'], 'content']!r}  ->  {r['text']!r}")

In [ ]:
_parts = []
for name in CZ_ALL_MODELS:
    d = cz_recs.copy()
    d['model'] = name
    d['margin_orig'] = cz_orig_margin[name][d['row'].values]
    d['margin_new'] = cz_margins(name, d['text'].tolist())
    _parts.append(d)
cz_int = pd.concat(_parts, ignore_index=True)

cz_int['reviewId'] = cz_test['reviewId'].values[cz_int['row'].values]
cz_int['label'] = cz_test['label'].values[cz_int['row'].values]
cz_int['pred_orig'] = (cz_int['margin_orig'] > 0).astype(int)
cz_int['pred_new'] = (cz_int['margin_new'] > 0).astype(int)
cz_int['flip'] = (cz_int['pred_orig'] != cz_int['pred_new']).astype(int)
cz_int['orig_correct'] = (cz_int['pred_orig'] == cz_int['label']).astype(int)
cz_int['new_correct'] = (cz_int['pred_new'] == cz_int['label']).astype(int)
cz_int['toward_true'] = (cz_int['margin_new'] - cz_int['margin_orig']) * np.where(cz_int['label'] == 1, 1, -1)
print("Intervention predictions computed:", len(cz_int))

#### Summary per model and token

Columns: `flip_delete` and `flip_replace` are the flip rates after deleting or replacing the trigger token; `flip_random` is the control (deleting another random word). `delete_minus_random` is the paired difference, and a confidence interval that excludes 0 means removing *this token* matters more than removing a random word. `repair_rate` is computed over originally wrong predictions (blank if there are none, e.g. XLM-RoBERTa) and `break_rate` over originally correct ones.

In [ ]:
_row = (cz_int.groupby(['model', 'token', 'kind', 'row', 'reviewId'])
        .agg(flip=('flip', 'mean'), orig_correct=('orig_correct', 'first'), new_correct=('new_correct', 'mean'))
        .reset_index())
_wide = _row.pivot_table(index=['model', 'token', 'row', 'reviewId', 'orig_correct'], columns='kind', values='flip').reset_index()
for c in ['delete_token', 'replace_token', 'delete_random', 'delete_cue']:
    if c not in _wide.columns:
        _wide[c] = np.nan
_wide['delete_minus_random'] = _wide['delete_token'] - _wide['delete_random']
_wide['replace_minus_random'] = _wide['replace_token'] - _wide['delete_random']

_nc = (cz_int[cz_int['kind'] == 'delete_token']
       .groupby(['model', 'token', 'row', 'reviewId'])
       .agg(orig_correct=('orig_correct', 'first'), new_correct=('new_correct', 'first')).reset_index())

_out = []
for (model, token), g in _wide[_wide['token'] != '(cue)'].groupby(['model', 'token']):
    gg = _nc[(_nc['model'] == model) & (_nc['token'] == token)]
    wrong, right = gg[gg['orig_correct'] == 0], gg[gg['orig_correct'] == 1]
    rec = {'model': model, 'token': token, 'n_rows': len(g), 'n_reviews': g['reviewId'].nunique(),
           'n_orig_wrong': len(wrong),
           'repair_rate': float(wrong['new_correct'].mean()) if len(wrong) else np.nan,
           'break_rate': float(1 - right['new_correct'].mean()) if len(right) else np.nan}
    for col in ['delete_token', 'replace_token', 'delete_random', 'delete_minus_random', 'replace_minus_random']:
        mean, lo, hi = cz_boot_mean_ci(g, col)
        rec[col] = mean; rec[col + '_lo'] = lo; rec[col + '_hi'] = hi
    _out.append(rec)
cz_int_summary = pd.DataFrame(_out)
cz_int_summary['model'] = pd.Categorical(cz_int_summary['model'], CZ_ALL_MODELS, ordered=True)
cz_int_summary = cz_int_summary.sort_values(['token', 'model']).reset_index(drop=True)

def _fmt(r, col):
    if pd.isna(r[col]):
        return ''
    return f"{r[col]:.2f} [{r[col + '_lo']:.2f}, {r[col + '_hi']:.2f}]"

_show = cz_int_summary[['model', 'token', 'n_rows', 'n_reviews', 'n_orig_wrong', 'repair_rate', 'break_rate']].copy()
for col, label in [('delete_token', 'flip_delete'), ('replace_token', 'flip_replace'),
                   ('delete_random', 'flip_random'), ('delete_minus_random', 'delete_minus_random')]:
    _show[label] = cz_int_summary.apply(lambda r: _fmt(r, col), axis=1)
_show[['repair_rate', 'break_rate']] = _show[['repair_rate', 'break_rate']].round(2)

print(f"Primary token: '{CZ_PRIMARY_TOKEN}'")
display(_show[_show['token'] == CZ_PRIMARY_TOKEN].drop(columns='token').reset_index(drop=True))
print("All trigger tokens (tokens present in fewer than ~10 reviews are underpowered):")
display(_show.reset_index(drop=True))

#### Positive control: deleting the sentiment cue itself

If a model has learned to read the cue, deleting it should change predictions much more often than deleting an unrelated word. A low flip rate here would mean the model barely uses the cue at all.

In [ ]:
_cue = _wide[_wide['token'] == '(cue)']
_o = []
for model, g in _cue.groupby('model'):
    mean, lo, hi = cz_boot_mean_ci(g, 'delete_cue')
    _o.append({'model': model, 'n_rows': len(g), 'n_reviews': g['reviewId'].nunique(),
               'flip_delete_cue': mean, 'ci_lo': lo, 'ci_hi': hi})
cz_cue_summary = pd.DataFrame(_o)
cz_cue_summary['model'] = pd.Categorical(cz_cue_summary['model'], CZ_ALL_MODELS, ordered=True)
cz_cue_summary = cz_cue_summary.sort_values('model').reset_index(drop=True)
cz_cue_summary[['flip_delete_cue', 'ci_lo', 'ci_hi']] = cz_cue_summary[['flip_delete_cue', 'ci_lo', 'ci_hi']].round(2)
cz_cue_summary

#### The paper's anchor example: "Cukup membingungkan aplikasi ini awal awal y."

Table 7 of the paper is built on this review (true label Negative). The table below shows, for each model and each of its six variants, the original prediction and the prediction after deleting "aplikasi". If "aplikasi" really drives the wrong prediction, the models that misclassified every variant (Logistic Regression, Linear SVC, BERT) should change their prediction here.

In [ ]:
_mask = np.array([(('membingungkan' in t.lower()) or ('confusing' in t.lower())) and cz_has_token(t, 'aplikasi')
                  for t in cz_test['content']], dtype=bool)
_anchor = cz_test[_mask]
if len(_anchor) == 0:
    print("The anchor review is not in this test set (check that the split was not changed).")
else:
    print(f"{len(_anchor)} variants found, true label(s): {sorted(_anchor['label'].unique())}  (0 = Negative)\n")

_tbl = []
for i, r in _anchor.iterrows():
    t_del = cz_delete_spans(r['content'], cz_target_spans(r['content'], 'aplikasi'))
    row = {'variant': r['content'], 'position': r['position'], 'language': r['language']}
    for name in CZ_ALL_MODELS:
        m0 = cz_orig_margin[name][i]
        m1 = cz_margins(name, [t_del])[0]
        row[name] = f"{'Pos' if m0 > 0 else 'Neg'} -> {'Pos' if m1 > 0 else 'Neg'}"
    _tbl.append(row)
cz_anchor_table = pd.DataFrame(_tbl)
cz_anchor_table

#### Graded effect: how far does the prediction margin move?

Flip rates are a coarse measure. A model can be pushed strongly towards a class by a token and still keep its label, because its margin was large to begin with. The measure below is the **absolute change of the positive-class margin** after an edit, in the model's own units (logit difference for transformers, `decision_function` for Logistic Regression and Linear SVC). Units differ between models, so only comparisons *within* a model are meaningful.

For the primary trigger token, on the same rows, three edits are compared: deleting the token, deleting one random other word (the control, averaged over the random draws), and deleting the sentiment cue. `token_minus_random` is the paired difference; an interval that excludes 0 means the token matters more than an arbitrary word. `token_over_cue` is the token's mean shift divided by the cue's mean shift: near 1 means the token moves the prediction about as much as the sentiment cue itself, near 0 means it barely matters.

In [ ]:
cz_int['abs_shift'] = (cz_int['margin_new'] - cz_int['margin_orig']).abs()

_p = (cz_int[cz_int['token'] == CZ_PRIMARY_TOKEN]
      .pivot_table(index=['model', 'row', 'reviewId'], columns='kind', values='abs_shift', aggfunc='mean')
      .reset_index())
_c = (cz_int[cz_int['kind'] == 'delete_cue'].groupby(['model', 'row'])['abs_shift'].mean()
      .rename('delete_cue').reset_index())
_p = _p.merge(_c, on=['model', 'row'], how='left')
_p['token_minus_random'] = _p['delete_token'] - _p['delete_random']
_p['cue_minus_random'] = _p['delete_cue'] - _p['delete_random']

_out = []
for model, g in _p.groupby('model'):
    rec = {'model': model, 'n_rows': len(g), 'n_reviews': g['reviewId'].nunique()}
    for col in ['delete_token', 'delete_random', 'delete_cue', 'token_minus_random', 'cue_minus_random']:
        mean, lo, hi = cz_boot_mean_ci(g, col)
        rec[col], rec[col + '_lo'], rec[col + '_hi'] = mean, lo, hi
    rec['token_over_cue'] = g['delete_token'].mean() / g['delete_cue'].mean()
    _out.append(rec)
cz_shift_summary = pd.DataFrame(_out)
cz_shift_summary['model'] = pd.Categorical(cz_shift_summary['model'], CZ_ALL_MODELS, ordered=True)
cz_shift_summary = cz_shift_summary.sort_values('model').reset_index(drop=True)

_show = cz_shift_summary[['model', 'n_rows', 'n_reviews']].copy()
for col in ['delete_token', 'delete_random', 'delete_cue', 'token_minus_random']:
    _show[col] = [f"{r[col]:.2f} [{r[col + '_lo']:.2f}, {r[col + '_hi']:.2f}]" for _, r in cz_shift_summary.iterrows()]
_show['token_over_cue'] = cz_shift_summary['token_over_cue'].round(2)
print(f"Mean absolute margin shift for '{CZ_PRIMARY_TOKEN}' rows (cluster-bootstrap 95% CI)")
display(_show)

## Part 4 - Integrated Gradients

Attention shows where a model *looks*; Integrated Gradients (IG) estimates how much each input token *contributes* to the prediction, which is what the review asked for. The attribution target is the positive-class margin (logit of Positive minus logit of Negative), the baseline is the same sentence with every non-special token replaced by padding, and attributions are summed over the embedding dimension for each token.

For each sentence and each token set (the trigger token, the sentiment cue) the cell reports:

- `share`: the set's share of the total absolute attribution over all non-special tokens;
- `share_over_uniform`: `share` divided by what an even spread would give (`pieces / non-special tokens`). Values above 1 mean the set receives more attribution than an average token set of its size;
- `toward_true`: the set's signed attribution, oriented so that positive values push towards the **true** label, divided by the total absolute attribution.

Convergence is checked through the completeness error; if it is large, raise `CZ_IG_STEPS`. Set `CZ_RUN_IG = False` in the configuration cell to skip this part.

In [ ]:
from tqdm.auto import tqdm

def cz_position_ids(name, ids):
    # Position ids of the REAL sentence. RoBERTa-style models (XLM-RoBERTa) derive position ids from the
    # input ids themselves (padding tokens get a fixed id). Without passing them explicitly, the all-padding
    # baseline would receive different positions than the real sentence, and the completeness check would
    # compare two different functions. BERT-style models use plain 0..n-1 positions.
    m = CZ_TRANSFORMERS[name]
    if m.config.model_type in ('roberta', 'xlm-roberta'):
        pad = m.config.pad_token_id
        keep = (ids != pad).long()
        return torch.cumsum(keep, dim=1) * keep + pad
    return torch.arange(ids.shape[1], device=ids.device).unsqueeze(0).expand_as(ids)


def cz_make_lig(name):
    model = CZ_TRANSFORMERS[name]
    model.eval()                                   # dropout would make the completeness check meaningless
    def fwd(input_ids, attention_mask, position_ids):
        lg = model(input_ids=input_ids, attention_mask=attention_mask, position_ids=position_ids).logits
        return lg[:, 1] - lg[:, 0]
    return LayerIntegratedGradients(fwd, model.get_input_embeddings()), fwd


def cz_ig_row(name, lig, fwd, text):
    tok = CZ_TOK[name]
    enc = tok(text, return_tensors='pt', return_offsets_mapping=True, truncation=True, max_length=CZ_MAX_LEN)
    offsets = enc.pop('offset_mapping')[0].tolist()
    ids = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)
    base = torch.full_like(ids, tok.pad_token_id)
    base[:, 0] = ids[:, 0]
    base[:, -1] = ids[:, -1]
    pos = cz_position_ids(name, ids)
    attr, delta = lig.attribute(ids, baselines=base, additional_forward_args=(mask, pos),
                                n_steps=CZ_IG_STEPS, internal_batch_size=16, return_convergence_delta=True)
    with torch.no_grad():
        gap = (fwd(ids, mask, pos) - fwd(base, mask, pos)).item()       # f(x) - f(baseline), what IG should add up to
    a = attr.sum(dim=-1)[0].detach().cpu().numpy()
    return a, offsets, ids[0].tolist(), float(delta.abs().item()), float(abs(gap))


cz_ig_records, cz_ig_conv = [], []
if CZ_RUN_IG:
    for name in CZ_TRANSFORMERS:
        lig, fwd = cz_make_lig(name)
        tok = CZ_TOK[name]
        skip = {tok.cls_token_id, tok.sep_token_id, tok.pad_token_id}
        preds = CZ_STORED_PREDS[name]
        for i, r in tqdm(list(cz_test.iterrows()), desc=f"IG {name}"):
            tsp = cz_target_spans(r['content'], CZ_PRIMARY_TOKEN)
            has_cue = isinstance(r['cue_spans'], list)
            if not tsp and not has_cue:
                continue                                   # nothing to compare in this sentence
            a, offsets, ids, dlt, gap = cz_ig_row(name, lig, fwd, r['content'])
            cz_ig_conv.append({'model': name, 'abs_delta': dlt, 'abs_gap': gap})
            rel_err = dlt / max(gap, 1e-6)
            valid = [k for k, t in enumerate(ids) if t not in skip]
            total = float(np.abs(a[valid]).sum())
            if total <= 0:
                continue
            sets = {}
            if tsp:
                sets['trigger_token'] = cz_token_idx(offsets, tsp)
            if has_cue:
                sets['sentiment_cue'] = cz_token_idx(offsets, r['cue_spans'])
            sign = 1.0 if r['label'] == 1 else -1.0
            for set_name, idx in sets.items():
                idx = [k for k in idx if k in set(valid)]
                if not idx:
                    continue
                share = float(np.abs(a[idx]).sum() / total)
                cz_ig_records.append({
                    'model': name, 'row': i, 'reviewId': r['reviewId'], 'set': set_name, 'pieces': len(idx),
                    'share': share, 'share_over_uniform': share / (len(idx) / len(valid)),
                    'toward_true': float(sign * a[idx].sum() / total),
                    'outcome': 'correct' if preds[i] == r['label'] else 'error',
                    'rel_err': rel_err, 'steps': CZ_IG_STEPS})
        del lig, fwd
        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    cz_ig = pd.DataFrame(cz_ig_records)
    _conv = pd.DataFrame(cz_ig_conv)
    _conv['rel_err'] = _conv['abs_delta'] / _conv['abs_gap'].clip(lower=1e-6)
    print("\nIG completeness check (median relative error per model; below ~0.05 is good):")
    print(_conv.groupby('model')['rel_err'].median().round(3).to_string())
else:
    print("CZ_RUN_IG is False: Part 4 skipped.")

#### Part 4b - Re-running Integrated Gradients where it has not converged

The check above is the honest test of whether the attributions can be used. A median relative error near 0 means the attributions add up to the change in the prediction; a value close to 1 means they do not, and the shares and signs for that model are not reliable. Two error measures are recorded. `rel_err` divides the error by `|f(x) - f(baseline)|`, which is what the first check printed; it becomes unstable when the baseline prediction is close to the real prediction (the gap is near 0), as can happen for a model that predicts one class for an all-padding input. `err_total` divides the error by the total absolute attribution, which is the quantity the reported shares and signs are normalised by, so it is the one that decides whether a row is usable. Models above `CZ_IG_TOL` on the first measure are re-run here with more interpolation steps, row by row, until the row's `err_total` is below `CZ_IG_ROW_TOL` or the top of the ladder is reached. Rows that still do not converge (`err_total` above `CZ_IG_DROP`) are marked `converged = False` and left out of the IG summary, and the number of rows left out is printed. If a model still does not converge, do not report its IG numbers; use the intervention results (Part 3) for that model instead.

In [ ]:
CZ_IG_TOL = 0.05                  # models with a median relative error above this are re-run
CZ_IG_LADDER = [64, 128, 256]     # interpolation steps tried in turn
CZ_IG_ROW_TOL = 0.10              # stop increasing the steps once a row is below this
CZ_IG_DROP = 0.20                 # rows still above this after the last step are excluded from the summary
CZ_IG_FORCE = ['XLM-RoBERTa']     # always re-run: its first run was computed without explicit position ids (see the text above)


# Re-define the position-aware helpers here, so that this cell alone is enough in a session that already ran Part 4.
def cz_position_ids(name, ids):
    # Position ids of the REAL sentence. RoBERTa-style models (XLM-RoBERTa) derive position ids from the
    # input ids themselves (padding tokens get a fixed id). Without passing them explicitly, the all-padding
    # baseline would receive different positions than the real sentence, and the completeness check would
    # compare two different functions. BERT-style models use plain 0..n-1 positions.
    m = CZ_TRANSFORMERS[name]
    if m.config.model_type in ('roberta', 'xlm-roberta'):
        pad = m.config.pad_token_id
        keep = (ids != pad).long()
        return torch.cumsum(keep, dim=1) * keep + pad
    return torch.arange(ids.shape[1], device=ids.device).unsqueeze(0).expand_as(ids)


def cz_make_lig(name):
    model = CZ_TRANSFORMERS[name]
    model.eval()
    def fwd(input_ids, attention_mask, position_ids):
        lg = model(input_ids=input_ids, attention_mask=attention_mask, position_ids=position_ids).logits
        return lg[:, 1] - lg[:, 0]
    return LayerIntegratedGradients(fwd, model.get_input_embeddings()), fwd


def cz_ig_row_steps(name, lig, fwd, text, n_steps):
    tok = CZ_TOK[name]
    enc = tok(text, return_tensors='pt', return_offsets_mapping=True, truncation=True, max_length=CZ_MAX_LEN)
    offsets = enc.pop('offset_mapping')[0].tolist()
    ids = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)
    base = torch.full_like(ids, tok.pad_token_id)
    base[:, 0] = ids[:, 0]
    base[:, -1] = ids[:, -1]
    pos = cz_position_ids(name, ids)
    attr, delta = lig.attribute(ids, baselines=base, additional_forward_args=(mask, pos),
                                n_steps=n_steps, internal_batch_size=16, return_convergence_delta=True)
    with torch.no_grad():
        gap = (fwd(ids, mask, pos) - fwd(base, mask, pos)).item()
        f_x = fwd(ids, mask, pos).item()
    a = attr.sum(dim=-1)[0].detach().cpu().numpy()
    return a, offsets, ids[0].tolist(), float(delta.abs().item()), float(abs(gap)), float(f_x)


def cz_ig_collect(name, ladder):
    lig, fwd = cz_make_lig(name)
    tok = CZ_TOK[name]
    skip = {tok.cls_token_id, tok.sep_token_id, tok.pad_token_id}
    preds = CZ_STORED_PREDS[name]
    recs = []
    for i, r in tqdm(list(cz_test.iterrows()), desc=f"IG refine {name}"):
        tsp = cz_target_spans(r['content'], CZ_PRIMARY_TOKEN)
        has_cue = isinstance(r['cue_spans'], list)
        if not tsp and not has_cue:
            continue
        for steps in ladder:
            a, offsets, ids, dlt, gap, f_x = cz_ig_row_steps(name, lig, fwd, r['content'], steps)
            valid = [k for k, t in enumerate(ids) if t not in skip]
            total = float(np.abs(a[valid]).sum())
            err_total = dlt / max(total, 1e-6)             # error relative to the total attribution that the shares are normalised by
            if err_total <= CZ_IG_ROW_TOL:
                break
        rel_err = dlt / max(gap, 1e-6)                      # error relative to f(x) - f(baseline): unstable when that gap is small
        if total <= 0:
            continue
        sets = {}
        if tsp:
            sets['trigger_token'] = cz_token_idx(offsets, tsp)
        if has_cue:
            sets['sentiment_cue'] = cz_token_idx(offsets, r['cue_spans'])
        sign = 1.0 if r['label'] == 1 else -1.0
        for set_name, idx in sets.items():
            idx = [k for k in idx if k in set(valid)]
            if not idx:
                continue
            share = float(np.abs(a[idx]).sum() / total)
            recs.append({'model': name, 'row': i, 'reviewId': r['reviewId'], 'set': set_name, 'pieces': len(idx),
                         'share': share, 'share_over_uniform': share / (len(idx) / len(valid)),
                         'toward_true': float(sign * a[idx].sum() / total),
                         'outcome': 'correct' if preds[i] == r['label'] else 'error',
                         'rel_err': rel_err, 'err_total': err_total, 'steps': steps, 'abs_gap': gap, 'abs_f': abs(f_x)})
    del lig, fwd
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return recs


if CZ_RUN_IG:
    _conv0 = pd.DataFrame(cz_ig_conv)
    _conv0['rel_err'] = _conv0['abs_delta'] / _conv0['abs_gap'].clip(lower=1e-6)
    _med0 = _conv0.groupby('model')['rel_err'].median()
    _bad = [m for m in CZ_TRANSFORMERS if _med0.get(m, 0) > CZ_IG_TOL or m in CZ_IG_FORCE]
    print("Median relative error before refinement:", _med0.round(3).to_dict())
    print("Models to re-run:", _bad if _bad else "none (all converged)")

    cz_ig = cz_ig.copy()
    for col, default in [('rel_err', np.nan), ('err_total', np.nan), ('steps', CZ_IG_STEPS)]:
        if col not in cz_ig.columns:
            cz_ig[col] = default
    for name in _bad:
        new = pd.DataFrame(cz_ig_collect(name, CZ_IG_LADDER))
        cz_ig = pd.concat([cz_ig[cz_ig['model'] != name], new], ignore_index=True)
        print(f"{name}: median |f(x)| = {new['abs_f'].median():.2f}, median |f(x) - f(baseline)| = {new['abs_gap'].median():.2f}  "
              f"(when the gap is close to 0, the first error measure is unstable; the second is not)")
    cz_ig['converged'] = ~(cz_ig['err_total'] > CZ_IG_DROP)     # rows without a recorded error (models that did not need a re-run) are kept
    print("\nAfter refinement, per model:")
    print("  rel_err = error / |f(x) - f(baseline)|;  err_total = error / total absolute attribution (the one the shares depend on)")
    for m, g in cz_ig.groupby('model'):
        print(f"  {m:12s} median rel_err = {g['rel_err'].median():.3f}   median err_total = {g['err_total'].median():.3f}   kept {int(g['converged'].sum())}/{len(g)}")
else:
    print("CZ_RUN_IG is False: Part 4b skipped.")

In [ ]:
if CZ_RUN_IG:
    cz_ig_ok = cz_ig[cz_ig['converged']] if 'converged' in cz_ig.columns else cz_ig
    cz_ig_summary = cz_summarize(cz_ig_ok, ['model', 'set', 'outcome'], ['share_over_uniform', 'toward_true'])
    cz_ig_summary[['mean', 'median', 'ci_lo', 'ci_hi']] = cz_ig_summary[['mean', 'median', 'ci_lo', 'ci_hi']].round(3)
    print("Attribution received by the trigger token / sentiment cue, relative to an even spread (1.0 = average token)")
    display(cz_ig_summary[cz_ig_summary['metric'] == 'share_over_uniform'].drop(columns='metric').reset_index(drop=True))
    print("Signed contribution towards the TRUE label (positive = helps the correct answer, negative = pushes towards the wrong one)")
    display(cz_ig_summary[cz_ig_summary['metric'] == 'toward_true'].drop(columns='metric').reset_index(drop=True))

## Export

In [ ]:
_exports = {
    'cz_token_stats.csv': cz_token_stats,
    'cz_lr_svc_coefficients.csv': cz_coef_df,
    'cz_attention_rows.csv': cz_att,
    'cz_attention_summary.csv': cz_att_summary,
    'cz_intervention_rows.csv': cz_int.drop(columns=['text']),
    'cz_intervention_summary.csv': cz_int_summary,
    'cz_cue_deletion_summary.csv': cz_cue_summary,
    'cz_anchor_example.csv': cz_anchor_table,
    'cz_margin_shift_summary.csv': cz_shift_summary,
}
if CZ_RUN_IG:
    _exports['cz_ig_rows.csv'] = cz_ig
    _exports['cz_ig_summary.csv'] = cz_ig_summary
for fname, frame in _exports.items():
    frame.to_csv(fname, index=False)
    print("Saved:", fname, f"({len(frame)} rows)")

### Reading these results for the paper

- **Attention (Part 2).** Report the instance and review counts next to the 2.62-4.03 range. Compare `error` against `correct` rows: if the ratio for the trigger token is similar in both, attention to it cannot be offered as evidence for the errors, only as a description of a very common token. The earlier ratio measured the trigger token against the *average other token*; the `target_over_cue` rows are the ones that actually compare it with the sentiment cue, so the abstract wording should follow whichever of the two is used.
- **Interventions (Part 3).** The claim "the model relies on this token" is supported only where deleting or replacing it flips predictions *more* than deleting a random word (the `delete_minus_random` interval excludes 0), and where the cue-deletion control shows the model does respond to the cue. Where the interval includes 0, say that no effect was detected, and mention the sample size, since only 55 test reviews exist.
- **XLM-RoBERTa.** Because Part 3 and Part 4 do not need errors, they give XLM-RoBERTa the first behavioural comparison with the other models. Whether it flips less, the same or more is the evidence for or against the sentence "only XLM-RoBERTa does not show this pattern"; if it turns out to behave like the others, that sentence must change.
- **Token statistics (Part 1).** A trigger token that is common in both classes (`pct_positive_given_token` close to `base_rate_positive`, large `pct_of_train_reviews`) cannot be the training-set shortcut it was described as. Report the table as it is.
- **Integrated Gradients (Part 4).** Agreement between IG, attention and the interventions strengthens a claim; disagreement should be reported as a limitation, not hidden. IG and attention are both explanation methods, and only Part 3 changes the input and observes the prediction.
- **Limits that remain.** One split, one seed per model, 55 test reviews, six variants per review that are not independent, a cue location that is exact for Indonesian rows and heuristic for English rows, and a fixed set of trigger tokens chosen after looking at errors.